# Measure analysis

##### Imports and params

In [ ]:
from __future__ import annotations


from ast import literal_eval
from pathlib import Path
import os
import re
from typing import Dict, List, Tuple

import ipywidgets as widgets
import numpy as np
import pandas as pd
import pprint

import re
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib.patheffects as pe

In [ ]:
plt.rcParams.update({
"figure.figsize": (18, 10),
"axes.grid": True,
"grid.linestyle": ":",
"grid.alpha": 0.5,
"axes.titlesize": 14,
"axes.labelsize": 12,
"legend.fontsize": 10,
})

In [ ]:
PATH_RESULTS = Path("/home/loai/Documents/code/RSMLExtraction/measureresults_per_plant.csv")
PATH_TB_LOGS = Path("~/Documents/code/RSMLExtraction/Results/Training/Logs/").expanduser()

## Loading database

In [ ]:
df_plant_wid = pd.read_csv(PATH_RESULTS)
df_plant_wid["root_ids"] = df_plant_wid["root_ids"].apply(literal_eval)
# remove metric = apls
#df_plant_wid = df_plant_wid[df_plant_wid['metric'] != 'apls'].reset_index(drop=True)

In [ ]:

dict_last_epochs = {}
dict_last_epochs['Unet_dice'] = 261
dict_last_epochs['Unet_cldice_dice'] = 260
dict_last_epochs['Unet_bce_dice'] = 267
dict_last_epochs['Unet_bce'] = 263
dict_last_epochs['Segformer_dice'] = 300
dict_last_epochs['Segformer_bce_dice'] = 285
dict_last_epochs['Segformer_bce'] = 286

In [ ]:
# if model name has no numeric suffix, replace model name by model_name_last where last is taken from dict_last_epochs
def add_epoch_suffix(m):
    tail = m.split('_')[-1]
    if tail.isdigit():
        return m
    return f"{m}_{dict_last_epochs.get(m, tail)}"

df_plant_wid["model"] = df_plant_wid["model"].map(add_epoch_suffix)

In [ ]:
# order by model name and epoch number
df_plant_wide = df_plant_wid.sort_values(by=['model', 'split', 'box', 'time', 'root_ids', 'metric']).reset_index(drop=True)
df_plant_wide

In [ ]:
_df_grouped = (
df_plant_wid
.set_index(["model", "split", "box", "metric", "root_ids", "source", "time"]).sort_index(level="time")
)

MODELS: List[str] = _df_grouped.index.get_level_values("model").unique().tolist()
SPLITS: List[str] = _df_grouped.index.get_level_values("split").unique().tolist()
BOXES: List[str] = _df_grouped.index.get_level_values("box").unique().tolist()
METRICS: List[str] = _df_grouped.index.get_level_values("metric").unique().tolist()
SOURCES: List[str] = _df_grouped.index.get_level_values("source").unique().tolist()

pprint.pprint({
    "Models": MODELS,
    "Splits": SPLITS,
    "Boxes": BOXES,
    "Metrics": METRICS,
    "Sources": SOURCES
})

### Helpers de sélection

In [ ]:
SPLIT_TO_BOXES = {
    SPLITS[0]: BOXES[0:5],  # 0 à 4
    SPLITS[1]: BOXES[5:10], # 5 à 9
}
pprint.pprint(SPLIT_TO_BOXES)

del BOXES 
del SPLITS

In [ ]:
DefLevels = Tuple[str, str, str, str]

def get_root_ids(model: str, split: str, box: str, metric: str) -> List[Tuple[int, ...]]:
    return (
    _df_grouped
    .xs((model, split, box, metric), level=("model", "split", "box", "metric"))
    .index.get_level_values("root_ids").unique().tolist()
    )


def slice_df(model: str, split: str, box: str, metric: str, root: Tuple[int, ...]) -> pd.DataFrame:
    return (
    _df_grouped
    .xs((model, split, box, metric, root), level=("model", "split", "box", "metric", "root_ids"))
    .reset_index()
    .assign(time=lambda d: pd.to_numeric(d["time"], errors="coerce").astype(float))
    .sort_values("time")
    )

In [ ]:
model = MODELS[10]
split = list(SPLIT_TO_BOXES.keys())[0]
box = list(SPLIT_TO_BOXES[split])[4]
metric = METRICS[0]
root_ids = get_root_ids(model, split, box, metric)
print(root_ids)
root = root_ids[0]
sliced_df = slice_df(model, split, box, metric, root)

pprint.pprint({
    "model": model,
    "split": split,
    "box": box,
    "metric": metric,
    "root": root,
    "root_ids": root_ids,
    "sliced_df": sliced_df
})

In [ ]:
STYLE_MAP = {
"before_expertized": dict(color="tab:orange", linestyle="--", marker=None, linewidth=2.2, markersize=7, alpha=0.8, xoff=-0.0),
"expertized": dict(color="tab:green", linestyle="-", marker="o", linewidth=2.4, markersize=7, alpha=0.9, xoff=0.00),
"Prediction": dict(color="tab:red", linestyle="-", marker="x", linewidth=2.2, markersize=7, alpha=0.9, xoff=+0.00),
}

def plot_root(model: str, split: str, box_idx: int, metric: str, root_idx: int):
    boxes_by_split = SPLIT_TO_BOXES.get(split, [])
    # Exemple original utilisait: box = boxes[box_idx + split_idx * 5]
    box = boxes_by_split[box_idx]


    roots = get_root_ids(model, split, box, metric)
    if len(roots) == 0:
        print("Aucune racine pour cette combinaison.")
        return
    root = roots[min(max(root_idx, 0), len(roots)-1)]


    df_tmp = slice_df(model, split, box, metric, root)


    plt.figure(figsize=(16, 8))
    for source, style in STYLE_MAP.items():
        d = df_tmp[df_tmp["source"] == source]
        if d.empty:
            continue
        x = d["time"]
        plt.plot(
        x, d["value"],
        label=source,
        color=style["color"], linestyle=style["linestyle"], marker=style["marker"],
        linewidth=style["linewidth"], markersize=style["markersize"], alpha=style["alpha"],
        )
        # Annotation du dernier point
        plt.text(x.iloc[-1], d["value"].iloc[-1], f" {source}", va="center", fontsize=9, color=style["color"])


    xt = sorted(df_tmp["time"].unique())
    plt.xticks(xt, xt)
    plt.title(f"{metric} — {model} / {split} / {box} / root {root}")
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.legend(title="Source")
    plt.tight_layout()
    plt.show()


w_model = widgets.Dropdown(options=MODELS, value=MODELS[0], description="Model")
w_split = widgets.Dropdown(options=list(SPLIT_TO_BOXES.keys()), value=list(SPLIT_TO_BOXES.keys())[0], description="Split")
w_box = widgets.IntSlider(min=0, max=4, step=1, value=0, description="Box idx") 
w_metric= widgets.Dropdown(options=METRICS, value=METRICS[0], description="Metric")
w_root = widgets.IntSlider(min=0, max=len(get_root_ids(w_model.value, w_split.value, list(SPLIT_TO_BOXES[w_split.value])[w_box.value], w_metric.value)) - 1, step=1, value=0, description="Root idx")

In [ ]:
def _update_root_slider(*_):
    try:
        roots = get_root_ids(w_model.value, w_split.value,
                             SPLIT_TO_BOXES[w_split.value][w_box.value], w_metric.value)
    except Exception:
        roots = []
    w_root.max = max(len(roots) - 1, 0)
    if w_root.value > w_root.max:
        w_root.value = w_root.max


for wid in (w_model, w_split, w_box, w_metric):
    wid.observe(_update_root_slider, "value")
_update_root_slider()


widgets.VBox([
    widgets.HBox([w_model, w_split, w_box, w_metric, w_root]),
    widgets.interactive_output(
        plot_root,
        dict(model=w_model, split=w_split, box_idx=w_box,
             metric=w_metric, root_idx=w_root)
    )
])

### Normalization 

In [ ]:
DfNorm = df_plant_wid.copy()
max_per_metric = DfNorm.groupby("metric")["value"].max().rename("max_value")
pprint.pprint(max_per_metric)

In [ ]:
DfNorm = DfNorm.merge(max_per_metric, on="metric", how="left")
DfNorm["value_norm"] = DfNorm["value"].astype(float) / (DfNorm["max_value"].astype(float))
DfNorm = DfNorm.drop(columns=["max_value"])
DfNorm

## Moyennage

In [ ]:
DfMean = (
_df_grouped.reset_index()
.groupby(["model", "metric", "time", "source"], as_index=False)
.agg(mean_value=("value", "mean"), std_value=("value", "std"))
)
DfMean

In [ ]:
def plot_mean_evolution(model: str, metric: str):
    d = DfMean[(DfMean["model"] == model) & (DfMean["metric"] == metric)].copy()
    if d.empty:
        print("Aucune donnée.")
        return
    d["time"] = pd.to_numeric(d["time"], errors="coerce")
    d = d.sort_values(["source", "time"])


    plt.figure(figsize=(14, 8))
    for source, style in (
    ("Prediction", {"color": "tab:red", "linestyle": "-", "marker": "x"}),
    ("before_expertized", {"color": "tab:orange", "linestyle": "--", "marker": None}),
    ("expertized", {"color": "tab:green", "linestyle": "-", "marker": "o"}),
    ):
        ds = d[d["source"] == source]
        if ds.empty:
            continue
        plt.errorbar(ds["time"], ds["mean_value"], yerr=ds["std_value"], label=source,
        linewidth=2.2, markersize=7, alpha=0.9, **{k:v for k,v in style.items() if k in ("linestyle","marker","color")})
    plt.title(f"Mean evolution — {model} / {metric}")
    plt.xlabel("Time"); plt.ylabel("Mean value")
    plt.legend(title="Source")
    plt.tight_layout(); plt.show()


widgets.interactive(
plot_mean_evolution,
model=widgets.Dropdown(options=MODELS, value=MODELS[0]),
metric=widgets.Dropdown(options=METRICS, value=METRICS[0])
)

## Error abs and signed

In [ ]:
DfPivot = (
    df_plant_wid
    .pivot_table(index=["model", "metric", "time", "split", "box", "root_ids"], columns="source", values="value")
    .reset_index()
)
DfPivot

In [ ]:
DfError = DfPivot.copy()
DfError["abs_error"] = (DfError.get("expertized") - DfError.get("Prediction")).abs()
DfError["real_error"] = (DfError.get("expertized") - DfError.get("Prediction"))
DfError = DfError[["model", "metric", "time", "split", "box", "root_ids", "abs_error", "real_error"]]

DfError

In [ ]:
DfMeanErr = (
    DfError.groupby(["model", "metric", "time"], as_index=False) # val and test set combined - IMPORTANT NOTE
    .agg(abs_error_mean=("abs_error", "mean"), real_error_mean=("real_error", "mean"), std_abs_error=("abs_error", "std"))
)
DfMeanErr

In [ ]:
max_val_metric = (
    DfMean[DfMean["source"] == "expertized"].groupby("metric")[
        "mean_value"].max()
)
pprint.pprint(max_val_metric)

In [ ]:
DfNormErr = DfMeanErr.copy() # NOTE : it is a normalized error dataframe by max value per metric
DfNormErr["max_value"] = DfNormErr.set_index(["metric"]).index.map(max_val_metric) # map max value per metric
DfNormErr["norm_abs_approx"] = 1.0 - DfNormErr["abs_error_mean"].astype(float) / DfNormErr["max_value"].astype(float) # 1 - (abs error_mean / max value)
DfNormErr["real_error_mean"] = DfNormErr["real_error_mean"].astype(float) / DfNormErr["max_value"].astype(float) # 1 - (real error_mean / max value) (sign counts)
DfNormErr["std_abs_error"] = DfNormErr["std_abs_error"].astype(float) / DfNormErr["max_value"].astype(float) # 1 - (std abs error / max value)
DfNormErr = DfNormErr.drop(columns=["max_value"]) # remove max_value column
DfNormErr

In [ ]:
ALL_MODELS_ERR = sorted(DfMeanErr["model"].unique().tolist())

def plot_distance_all_models(metric: str):
    d = DfMeanErr[DfMeanErr["metric"] == metric].copy()
    if d.empty:
        print(f"Aucune donnée pour {metric}.")
        return
    d["time"] = pd.to_numeric(d["time"], errors="coerce")
    d = d.sort_values(["model", "time"]) 

    plt.figure(figsize=(14, 7))
    for m in ALL_MODELS_ERR:
        dm = d[d["model"] == m]
        if dm.empty:
            continue
        plt.plot(dm["time"], dm["abs_error_mean"], marker="o", linewidth=2.0, markersize=6, alpha=0.95, label=m)

    plt.title(f"Évolution de l'erreur absolue moyenne — métrique: {metric}")
    plt.xlabel("Time"); plt.ylabel("Mean absolute error")
    plt.legend(title="Model x loss", ncol=5)
    plt.tight_layout(); plt.show()

widgets.interactive(plot_distance_all_models, metric=widgets.Dropdown(options=METRICS, value=METRICS[0]))

## Helpers

In [ ]:
# -- Helpers
_epoch_re = re.compile(r'_(\d+)$')

def base_name(model: str) -> str:
    # "Unet_bce_060" -> "Unet" ; "Segformer_bce_dice_140" -> "Segformer"
    return model[:model.rfind('_')]

def extract_epochs(model: str):
    m = _epoch_re.search(model)
    return int(m.group(1)) if m else None

def blend_with_white(color, intensity: float):
    """
    intensity in [0,1]; 0 -> blanc, 1 -> couleur d'origine.
    """
    c = np.array(mcolors.to_rgb(color))
    return tuple((1 - intensity) * 1.0 + intensity * c)

# Palette VIVE (couleurs marquées)
BOLD_COLORS = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
    "#a65628", "#f781bf", "#999999", "#66c2a5", "#a6cee3",
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e",
]

ALL_MODELS_ERR = sorted(DfMeanErr["model"].unique().tolist())
print(ALL_MODELS_ERR)

# keep only : 'Segformer_bce_286', 'Segformer_bce_dice_285', 'Segformer_dice_300', 'Unet_bce_263', 'Unet_bce_dice_267', 'Unet_cldice_dice_260', 'Unet_dice_261'
ALL_MODELS_ERR = [
    'Segformer_bce_286', 'Segformer_bce_dice_285', 'Segformer_dice_300',
    'Unet_bce_263', 'Unet_bce_dice_267', 'Unet_cldice_dice_260', 'Unet_dice_261'
]

def plot_distance_all_models(metric: str, use_error_bars: bool = True):
    d = DfMeanErr[DfMeanErr["metric"] == metric].copy()
    if d.empty:
        print(f"Aucune donnée pour {metric}.")
        return

    # Prépare colonnes
    d["time"] = pd.to_numeric(d["time"], errors="coerce")
    d["base"] = d["model"].map(base_name)
    d["epochs"] = d["model"].map(extract_epochs)
    d = d.sort_values(["base", "model", "time"])

    # Teinte par famille (couleurs marquées)
    bases = d["base"].dropna().unique().tolist()
    base2color = {b: BOLD_COLORS[i % len(BOLD_COLORS)] for i, b in enumerate(bases)}

    # Intensité par #epochs, normalisée par famille
    inten_min, inten_max = 0.45, 1.00
    base2norm = {}
    for b in bases:
        e = d.loc[d["base"] == b, "epochs"].dropna().values
        if len(e) == 0:
            base2norm[b] = (None, None, None)
        else:
            e_min, e_max = np.min(e), np.max(e)
            span = max(1, e_max - e_min)
            base2norm[b] = (e_min, e_max, span)

    plt.figure(figsize=(14, 7))

    already_labeled = set()
    for m in ALL_MODELS_ERR:
        dm = d[d["model"] == m]
        if dm.empty:
            continue

        b = base_name(m)
        base_col = base2color.get(b, "#377eb8")
        ep = extract_epochs(m)

        if ep is None or base2norm[b][0] is None:
            intensity = 0.75
        else:
            e_min, _, span = base2norm[b]
            norm = (ep - e_min) / span  # 0..1
            intensity = inten_min + (inten_max - inten_min) * norm

        line_color = blend_with_white(base_col, intensity)

        label = b if b not in already_labeled else None
        if label is not None:
            already_labeled.add(b)

        # Ligne bien lisible + marqueurs contrastés
        plt.plot(
            dm["time"], dm["abs_error_mean"],
            marker="o", markersize=6, markerfacecolor="white",
            markeredgewidth=1.6, markeredgecolor=line_color,
            linewidth=2.6, solid_capstyle="round", solid_joinstyle="round",
            alpha=0.98, color=line_color, label=label, zorder=3
        )


    # Légende compacte: une par famille (teinte quasi pleine)
    legend_handles = []
    for b in bases:
        c = blend_with_white(base2color[b], 0.96)
        legend_handles.append(Line2D([0], [0], color=c, lw=4, label=b))
    plt.legend(handles=legend_handles, title="Famille de modèle", ncol=2, frameon=True)

    plt.title(f"Evolution of the number of detected organs of the plant for each model")
    plt.xlabel("Time"); plt.ylabel("Mean absolute error")
    plt.grid(True, axis="y", alpha=0.25, linewidth=0.8)
    plt.tight_layout()
    plt.show()

# Widget : vous pouvez basculer bandes/barres d'erreur en jouant avec use_error_bars
widgets.interactive(
    plot_distance_all_models,
    metric=widgets.Dropdown(options=METRICS, value=METRICS[0]),
    use_error_bars=widgets.Checkbox(value=True, description="Barres d'erreur (lisibles)")
)


In [ ]:
def plot_distance_all_models(metric: str, use_error_bars: bool = True):
    d = DfMeanErr[DfMeanErr["metric"] == metric].copy()
    if d.empty:
        print(f"Aucune donnée pour {metric}.")
        return

    # Prépare colonnes
    d["time"] = pd.to_numeric(d["time"], errors="coerce")
    d["base"] = d["model"].map(base_name)
    d["epochs"] = d["model"].map(extract_epochs)
    d = d.sort_values(["base", "model", "time"])

    # Teinte par famille (couleurs marquées)
    bases = d["base"].dropna().unique().tolist()
    base2color = {b: BOLD_COLORS[i % len(BOLD_COLORS)] for i, b in enumerate(bases)}

    # Intensité par #epochs, normalisée par famille
    inten_min, inten_max = 0.45, 1.00
    base2norm = {}
    for b in bases:
        e = d.loc[d["base"] == b, "epochs"].dropna().values
        if len(e) == 0:
            base2norm[b] = (None, None, None)
        else:
            e_min, e_max = np.min(e), np.max(e)
            span = max(1, e_max - e_min)
            base2norm[b] = (e_min, e_max, span)

    plt.figure(figsize=(14, 7))

    already_labeled = set()
    for m in ALL_MODELS_ERR:
        dm = d[d["model"] == m]
        if dm.empty:
            continue

        b = base_name(m)
        base_col = base2color.get(b, "#377eb8")
        ep = extract_epochs(m)

        if ep is None or base2norm[b][0] is None:
            intensity = 0.75
        else:
            e_min, _, span = base2norm[b]
            norm = (ep - e_min) / span  # 0..1
            intensity = inten_min + (inten_max - inten_min) * norm

        line_color = blend_with_white(base_col, intensity)

        label = b if b not in already_labeled else None
        if label is not None:
            already_labeled.add(b)

        # Ligne bien lisible + marqueurs contrastés
        plt.plot(
            dm["time"], dm["real_error_mean"],
            marker="o", markersize=6, markerfacecolor="white",
            markeredgewidth=1.6, markeredgecolor=line_color,
            linewidth=2.6, solid_capstyle="round", solid_joinstyle="round",
            alpha=0.98, color=line_color, label=label, zorder=3
        )


    # Légende compacte: une par famille (teinte quasi pleine)
    legend_handles = []
    for b in bases:
        c = blend_with_white(base2color[b], 0.96)
        legend_handles.append(Line2D([0], [0], color=c, lw=4, label=b))
    plt.legend(handles=legend_handles, title="Famille de modèle", ncol=2, frameon=True)

    plt.title(f"Evolution of the number of detected organs of the plant for each model - signed error")
    plt.xlabel("Time"); plt.ylabel("Mean absolute error")
    plt.grid(True, axis="y", alpha=0.25, linewidth=0.8)
    plt.tight_layout()
    plt.show()

# Widget : vous pouvez basculer bandes/barres d'erreur en jouant avec use_error_bars
widgets.interactive(
    plot_distance_all_models,
    metric=widgets.Dropdown(options=METRICS, value=METRICS[0]),
    use_error_bars=widgets.Checkbox(value=True, description="Barres d'erreur (lisibles)")
)


In [ ]:

def plot_distance_evolution(model: str, metric: str):
    d = DfNormErr[(DfNormErr["model"]==model) & (DfNormErr["metric"]==metric)].copy()
    if d.empty:
        print("Aucune paire Prediction/expertized pour cette combinaison.")
        return
    t = pd.to_numeric(d["time"], errors="coerce")
    plt.figure(figsize=(14, 7))
    plt.errorbar(t, d["norm_abs_approx"], yerr=d.get("std_abs_error"), linestyle='-', marker='o', linewidth=2.5, markersize=7, alpha=0.9)
    plt.ylim(0, 1.2)
    plt.title(f"Évolution de l'erreur absolue normalisée — {model} / {metric}")
    plt.xlabel("Time"); plt.ylabel("Norm. mean absolute error")
    plt.tight_layout(); plt.show()

widgets.interactive(
    plot_distance_evolution,
    model=widgets.Dropdown(options=MODELS, value=MODELS[0]),
    metric=widgets.Dropdown(options=METRICS, value=METRICS[0])
)


In [ ]:
def extract_last_metrics(base_path: Path) -> Dict[str, Dict[str, float]]:
    """Explore `base_path` -> {metric_name: {model_name: last_value}}"""
    metrics: Dict[str, Dict[str, float]] = {}
    for model_name in os.listdir(base_path):
        model_path = base_path / model_name / "logs" / "scalars"
        if not model_path.is_dir():
            continue
        for file in os.listdir(model_path):
            if not file.endswith(".csv"):
                continue
            fp = model_path / file
            try:
                df = pd.read_csv(fp)
                if "value" not in df.columns or df.empty:
                    continue
                last_val = float(df["value"].iloc[-1])
                metric_name = file.replace("val_", "").replace(".csv", "")
                metrics.setdefault(metric_name, {})[model_name] = last_val
            except Exception as e:
                print(f"⚠️ Lecture impossible {fp}: {e}")
    return metrics

def extract_metrics_epoch(base_path: Path, epoch_num: int) -> Dict[str, Dict[str, float]]:
    """Retourne les valeurs à l'époque `epoch_num` (indexée 1) si possible, sinon fallback last."""
    metrics: Dict[str, Dict[str, float]] = {}
    for model_name in os.listdir(base_path):
        model_path = base_path / model_name / "logs" / "scalars"
        if not model_path.is_dir():
            continue
        for file in os.listdir(model_path):
            if not file.endswith(".csv"):
                continue
            fp = model_path / file
            try:
                df = pd.read_csv(fp)
                if "value" not in df.columns or df.empty:
                    continue
                if epoch_num >= len(df):
                    print(f"⚠️ Pas assez d'époques dans {fp}")
                    return extract_last_metrics(base_path)
                step_idx = int(df["step"].iloc[epoch_num - 2])
                epoch_val = float(df["value"].iloc[step_idx])
                metric_name = file.replace("val_", "").replace(".csv", "")
                metrics.setdefault(metric_name, {})[model_name] = epoch_val
            except Exception as e:
                print(f"⚠️ Lecture impossible {fp}: {e}")
    return metrics

def extract_metrics_corresponding_epoch(base_path: Path, all_models: List[str]) -> Dict[str, Dict[str, float]]:
    """`all_models` comme ['Unet_bce_063', ...] -> aligne les époques par suffixe numérique _XYZ."""
    epochs = [int(m.split('_')[-1]) if m.split('_')[-1].isdigit() else -1 for m in all_models]
    base_names = [f"{m[:m.rfind('_')]}" for m in all_models]
    metrics: Dict[str, Dict[str, float]] = {}
    print(np.unique(base_names))

    for model_dir in os.listdir(base_path):
        model_path = base_path / model_dir / "logs" / "scalars"
        if not model_path.is_dir():
            continue
        if model_dir not in base_names:
            continue
        for ref_model, epoch_num in zip(all_models, epochs):
            if not ref_model[:ref_model.rfind('_')] == model_dir.split('/')[-1]:
                continue
            for file in os.listdir(model_path):
                if not file.endswith(".csv"):
                    continue
                fp = model_path / file
                try:
                    df = pd.read_csv(fp)
                    if "value" not in df.columns or df.empty:
                        continue
                    if epoch_num > len(df):
                        print(model_dir, model_path,ref_model, epoch_num)
                        print(f"⚠️ Pas assez d'époques dans {fp} pour epoch {epoch_num}")
                        print(df)
                        return extract_last_metrics(base_path)
                    
                    step_idx = int(df["step"].iloc[epoch_num - 2])
                    epoch_val = float(df["value"].iloc[step_idx])
                    metric_name = file.replace("val_", "").replace(".csv", "")
                    metrics.setdefault(metric_name, {})[ref_model] = epoch_val
                except Exception as e:
                    print(f"⚠️ Lecture impossible {fp}: {e}")
    return metrics

# --- Récupération valeurs CV pour les modèles présents côté graph ---
ALL_MODELS_GRAPH = sorted(DfMeanErr["model"].unique().tolist())
cv_metrics_results = extract_metrics_corresponding_epoch(PATH_TB_LOGS, all_models=ALL_MODELS_GRAPH)

# Jointure tableau récap pour scatter
DfCvGraph = (
    DfMeanErr.copy() # NOTE : from DfMeanErr
    .groupby(["metric","model"])[["abs_error_mean","std_abs_error"]] # IGNORE reel error, and mean over all times (sum of of the error of each model for each metric at all times)
    .mean().reset_index()
)
for metric_name, values_by_model in cv_metrics_results.items():
    DfCvGraph[metric_name] = DfCvGraph["model"].map(values_by_model)

DfCvGraph

In [ ]:
df_scatter = DfCvGraph.copy()
cv_metrics = list(cv_metrics_results.keys())
graph_metrics = df_scatter["metric"].unique().tolist()

models_all = df_scatter['model'].unique().tolist()

def _base_name(m: str) -> str:
    return m[:m.rfind('_')] if '_' in m else m

model_bases = {m: _base_name(m) for m in models_all}
base_names = sorted(np.unique(list(model_bases.values())).tolist())
groups_by_base = {b: [m for m in models_all if model_bases[m] == b] for b in base_names}
print(groups_by_base)

# Couleurs: une par base ; variantes teintées selon suffixe
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
base_color_map = {b: colors[i % len(colors)] for i, b in enumerate(base_names)}

def _suffix_num(name: str) -> int | None:
    m = re.search(r'_(\d{3})(?:\D|$)', name)
    return int(m.group(1)) if m else None

def _color_from_suffix(model_name: str, color: str = '#1f77b4'):
    m = re.search(r'_(\d{3})(?:\D|$)', model_name)
    v = int(m.group(1)) if m else 280
    v = max(60, min(280, v))
    t = (v - 40) / (300 - 40)
    color_to_text = {
        '#1f77b4': 'Blue', '#ff7f0e': 'Orange', '#2ca02c': 'Green', '#d62728': 'Red',
        '#9467bd': 'Purple', '#17becf': 'Cyan'
    }
    return cm.get_cmap(f"{color_to_text.get(color, 'Blue')}s_r")(t)

variant_color_map = {}
for m in models_all:
    base = model_bases[m]
    variant_color_map[m] = _color_from_suffix(m, color=base_color_map[base])

# --- Plot filtré par base + polyligne min→max ---
def plot_scatter_by_base(model_base: str,
                         cv_key: str,
                         graph_key: str,
                         show_error: bool=True,
                         alpha_pts: float=0.9,
                         size_pts: int=50,
                         show_labels: bool=True,
                         label_size: int=9,
                         show_track: bool=True,
                         with_arrow: bool=True,
                         track_lw: float=2.0,
                         track_alpha: float=0.9):
    if cv_key not in cv_metrics:
        print(f"Métrique '{cv_key}' inconnue. Choisis parmi: {cv_metrics}")
        return
    if graph_key not in graph_metrics:
        print(f"Métrique '{graph_key}' inconnue. Choisis parmi: {graph_metrics}")
        return

    d = df_scatter.dropna(subset=[cv_key, 'metric', 'abs_error_mean']).copy()
    d = d[d['metric'] == graph_key]

    # Ne garder que les variantes de la base choisie
    variants = groups_by_base.get(model_base, [])
    d = d[d['model'].isin(variants)]
    if d.empty:
        print("Nothing to see here")
        return

    fig = plt.figure(figsize=(12, 7))
    ax = plt.gca()

    # 1) Points / barres d'erreur / étiquettes
    for m in variants:
        dm = d[d['model'] == m]
        if dm.empty:
            continue
        x = dm[cv_key].values
        y = dm['abs_error_mean'].values
        c = variant_color_map[m]
        ax.scatter(x, y, s=size_pts, alpha=alpha_pts, label=m, color=c, zorder=3)

        if show_error and 'std_abs_error' in dm.columns:
            yerr = dm['std_abs_error'].values
            ax.errorbar(x, y, yerr=yerr, fmt='none', ecolor=c, alpha=0.5, capsize=2, zorder=2)

        if show_labels:
            x_min, x_max = ax.get_xlim(); y_min, y_max = ax.get_ylim()
            dx = 0.015 * (x_max - x_min); dy = 0.015 * (y_max - y_min)
            for xi, yi in zip(x, y):
                ax.annotate(
                    m, (xi, yi),
                    xytext=(dx, dy), textcoords='offset points',
                    fontsize=label_size, color=c, alpha=0.95, ha='left', va='bottom',
                    path_effects=[pe.withStroke(linewidth=2, foreground='white', alpha=0.9)],
                    zorder=4
                )

    # 2) Polyligne min→max sur la base
    if show_track:
        seq = []
        for m in variants:
            dm = d[d['model'] == m]
            if dm.empty:
                continue
            num = _suffix_num(m)
            if num is None:
                continue
            xi = dm[cv_key].values[0]
            yi = dm['abs_error_mean'].values[0]
            if not (np.isfinite(xi) and np.isfinite(yi)):
                continue
            seq.append((num, xi, yi))
        if len(seq) >= 2:
            seq.sort(key=lambda t: t[0])
            xs = [t[1] for t in seq]
            ys = [t[2] for t in seq]
            base_col = base_color_map[model_base]
            ax.plot(xs, ys, '-', linewidth=track_lw, alpha=track_alpha, color=base_col, zorder=1)
            if with_arrow and len(xs) >= 2:
                x0, y0 = xs[-2], ys[-2]; x1, y1 = xs[-1], ys[-1]
                if all(np.isfinite(v) for v in (x0,y0,x1,y1)):
                    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                                arrowprops=dict(arrowstyle='->', color=base_col,
                                                lw=track_lw, alpha=track_alpha),
                                zorder=2)

    ax.set_title(f"{model_base} — abs_error_mean vs {cv_key} (graph: {graph_key})")
    ax.set_xlabel(cv_key)
    ax.set_ylabel("abs_error_mean (plus bas = mieux)")
    ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout(); plt.show()

# --- Slider de modèle (base) ---
w_model_base = widgets.SelectionSlider(options=base_names, value=base_names[0], description="Model (base)")

widgets.interactive(
    plot_scatter_by_base,
    model_base=w_model_base,
    cv_key=widgets.Dropdown(options=cv_metrics, value=(cv_metrics[0] if cv_metrics else None), description="CV metric"),
    graph_key=widgets.Dropdown(options=graph_metrics, value=(graph_metrics[0] if graph_metrics else None), description="Graph error"),
    show_error=widgets.Checkbox(value=False, description="Barres d’erreur"),
    alpha_pts=widgets.FloatSlider(value=0.9, min=0.2, max=1.0, step=0.1, description="Alpha points"),
    size_pts=widgets.IntSlider(value=50, min=10, max=120, step=5, description="Taille points"),
    show_labels=widgets.Checkbox(value=True, description="Étiquettes"),
    label_size=widgets.IntSlider(value=9, min=6, max=16, step=1, description="Taille texte"),
    show_track=widgets.Checkbox(value=True, description="Polyligne min→max"),
    with_arrow=widgets.Checkbox(value=True, description="Flèche direction"),
    track_lw=widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1, description="Épaisseur ligne"),
    track_alpha=widgets.FloatSlider(value=0.9, min=0.2, max=1.0, step=0.05, description="Alpha ligne"),
)


In [ ]:
df_scatter = DfCvGraph.copy()

cv_metrics = list(cv_metrics_results.keys())
graph_metrics = df_scatter["metric"].unique().tolist()

models_all = df_scatter['model'].unique().tolist()
model_labels = [m[:m.rfind('_')] for m in models_all]
models_order = np.unique(model_labels).tolist()
models_order = [m.lower() for m in models_order]

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
color_map = {m: colors[i % len(colors)] for i, m in enumerate(models_order)}  # défaut

print("Couleurs de base:", color_map)


def _color_from_suffix(model_name: str, color: str = '#1f77b4'):
    m = re.search(r'_(\d{3})(?:\D|$)', model_name)
        
    v = int(m.group(1)) if m else 280
    v = max(60, min(260, v))
    t = (v - 40) / (500 - 40)
    color_to_text = {
        '#1f77b4': 'Blue', '#ff7f0e': 'Orange', '#2ca02c': 'Green', '#d62728': 'Red',
        '#9467bd': 'Purple', '#17becf': 'Cyan'
    }
    return cm.get_cmap(f"{color_to_text.get(color, 'Blue')}s_r")(t)

for m in models_all:
    ml = m.lower()
    color_map[m] = _color_from_suffix(m, color=color_map[ml[:ml.rfind('_')]])


def plot_scatter_norm_vs_external(cv_key: str, graph_key: str, show_error: bool=True, alpha_pts: float=0.9, size_pts: int=50,
                                  show_labels: bool=True, label_size: int=9):
    if cv_key not in cv_metrics:
        print(f"Métrique '{cv_key}' inconnue. Choisis parmi: {cv_metrics}")
        return
    if graph_key not in graph_metrics:
        print(f"Métrique '{graph_key}' inconnue. Choisis parmi: {graph_metrics}")
        return
    d = df_scatter.dropna(subset=[cv_key, 'metric', 'abs_error_mean']).copy()
    d = d[d['metric'] == graph_key]
    if d.empty:
        print("Nothing to see here")
        return

    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    for m in models_all:
        dm = d[d['model'] == m]
        if dm.empty:
            continue
        x = dm[cv_key].values
        y = dm['abs_error_mean'].values
        c = color_map[m]
        plt.scatter(x, y, s=size_pts, alpha=alpha_pts, label=m, color=c)
        if show_error and 'std_abs_error' in dm.columns:
            yerr = dm['std_abs_error'].values
            plt.errorbar(x, y, yerr=yerr, fmt='none', ecolor=c, alpha=0.5, capsize=2)
        if show_labels:
            x_min, x_max = ax.get_xlim(); y_min, y_max = ax.get_ylim()
            dx = 0.015 * (x_max - x_min); dy = 0.015 * (y_max - y_min)
            for xi, yi in zip(x, y):
                ax.annotate(m, (xi, yi), xytext=(dx, dy), textcoords='offset points', fontsize=label_size,
                            color=c, alpha=0.95, ha='left', va='bottom',
                            path_effects=[pe.withStroke(linewidth=2, foreground='white', alpha=0.9)])

    plt.title(f"Scatter abs_error_mean vs {cv_key} — graph metric: {graph_key}")
    plt.xlabel(cv_key); plt.ylabel("abs_error_mean (plus bas = mieux)")
    plt.grid(True, linestyle=":", alpha=0.5)
    # Légende optionnelle si vous préférez les étiquettes inline
    #plt.legend(title="Modèle", ncol=2)
    plt.tight_layout(); plt.show()

widgets.interactive(
    plot_scatter_norm_vs_external,
    cv_key=widgets.Dropdown(options=sorted(cv_metrics), value=(cv_metrics[0] if cv_metrics else None), description="CV metric"),
    graph_key=widgets.Dropdown(options=graph_metrics, value=(graph_metrics[0] if graph_metrics else None), description="Graph error"),
    show_error=widgets.Checkbox(value=False, description="Barres d’erreur"),
    alpha_pts=widgets.FloatSlider(value=0.9, min=0.2, max=1.0, step=0.1, description="Alpha"),
    size_pts=widgets.IntSlider(value=50, min=10, max=120, step=5, description="Taille points"),
    show_labels=widgets.Checkbox(value=True, description="Étiquettes"),
    label_size=widgets.IntSlider(value=9, min=6, max=16, step=1, description="Taille texte"),
)

In [ ]:
# --- Ajouts utilitaires ---
import re
import matplotlib.patheffects as pe

# Regroupe les variantes par "base" (tout sauf le suffixe _NNN)
model_bases = {m: (m[:m.rfind('_')] if '_' in m else m) for m in models_all}
groups_by_base = {}
for m, base in model_bases.items():
    groups_by_base.setdefault(base, []).append(m)

def _suffix_num(name: str) -> int | None:
    m = re.search(r'_(\d{3})(?:\D|$)', name)
    return int(m.group(1)) if m else None


def plot_scatter_norm_vs_external(cv_key: str,
                                  graph_key: str,
                                  show_error: bool = True,
                                  alpha_pts: float = 0.9,
                                  size_pts: int = 50,
                                  show_labels: bool = True,
                                  label_size: int =  9,  # 9 (évite un warning de style)
                                  show_tracks: bool = True,      # ← polylignes on/off
                                  with_arrows: bool = True,       # ← flèche fin → sens min→max
                                  track_lw: float = 1.8,
                                  track_alpha: float = 0.75):
    if cv_key not in cv_metrics:
        print(f"Métrique '{cv_key}' inconnue. Choisis parmi: {cv_metrics}")
        return
    if graph_key not in graph_metrics:
        print(f"Métrique '{graph_key}' inconnue. Choisis parmi: {graph_metrics}")
        return

    d = df_scatter.dropna(subset=[cv_key, 'metric', 'abs_error_mean']).copy()
    d = d[d['metric'] == graph_key]
    if d.empty:
        print("Nothing to see here")
        return

    fig = plt.figure(figsize=(12, 7))
    ax = plt.gca()

    # 1) Points + (option) barres d'erreur + étiquettes
    for m in models_all:
        dm = d[d['model'] == m]
        if dm.empty:
            continue
        x = dm[cv_key].values
        y = dm['abs_error_mean'].values
        c = color_map[m]
        ax.scatter(x, y, s=size_pts, alpha=alpha_pts, label=m, color=c, zorder=3)

        if show_error and 'std_abs_error' in dm.columns:
            yerr = dm['std_abs_error'].values
            ax.errorbar(x, y, yerr=yerr, fmt='none', ecolor=c, alpha=0.5, capsize=2, zorder=2)

        if show_labels:
            x_min, x_max = ax.get_xlim(); y_min, y_max = ax.get_ylim()
            dx = 0.015 * (x_max - x_min); dy = 0.015 * (y_max - y_min)
            for xi, yi in zip(x, y):
                ax.annotate(
                    m, (xi, yi),
                    xytext=(dx, dy), textcoords='offset points',
                    fontsize=label_size, color=c, alpha=0.95, ha='left', va='bottom',
                    path_effects=[pe.withStroke(linewidth=2, foreground='white', alpha=0.9)],
                    zorder=4
                )

    # 2) Polylignes par "base" (ordre min_num → max_num)
    if show_tracks:
        for base, variants in groups_by_base.items():
            seq = []
            for name in variants:
                dm = d[d['model'] == name]
                if dm.empty:
                    continue
                num = _suffix_num(name)
                if num is None:
                    continue
                xi = dm[cv_key].values[0]
                yi = dm['abs_error_mean'].values[0]
                if not (np.isfinite(xi) and np.isfinite(yi)):
                    continue
                seq.append((num, xi, yi, name))

            if len(seq) < 2:
                continue

            seq.sort(key=lambda t: t[0])  # tri par suffixe numérique croissant
            xs = [t[1] for t in seq]
            ys = [t[2] for t in seq]

            base_key = base.lower()
            base_col = color_map.get(base_key, 'tab:blue')

            # polyligne
            ax.plot(xs, ys, linestyle='-', linewidth=track_lw, alpha=track_alpha,
                    color=base_col, zorder=1)

            # flèche sur le dernier segment (direction min→max)
            if with_arrows and len(xs) >= 2:
                x0, y0 = xs[-2], ys[-2]
                x1, y1 = xs[-1], ys[-1]
                if np.isfinite(x0) and np.isfinite(y0) and np.isfinite(x1) and np.isfinite(y1):
                    ax.annotate(
                        '', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='->', color=base_col,
                                        lw=track_lw, alpha=track_alpha),
                        zorder=2
                    )

    ax.set_title(f"Scatter abs_error_mean vs {cv_key} — graph metric: {graph_key}")
    ax.set_xlabel(cv_key)
    ax.set_ylabel("abs_error_mean (plus bas = mieux)")
    ax.invert_yaxis()
    ax.grid(True, linestyle=":", alpha=0.5)
    #if cv_key == "Specificity" or cv_key == "Recall" or cv_key == "Precision" or cv_key == "F1Score":
        #ax.set_xlim(1, 1.05)
        
    if any(keyword in cv_key.lower() for keyword in ["distance", "error", "persistence", "variation", "loss"]):
        ax.invert_xaxis()

    if cv_key == "train_lr" or cv_key == "train_batch_loss":
        ax.set_xscale("log")
    
    
    plt.tight_layout()
    plt.show()


# Widgets (on ajoute les options pour la polyligne)
widgets.interactive(
    plot_scatter_norm_vs_external,
    cv_key=widgets.Dropdown(options=sorted(cv_metrics),
                            value=(cv_metrics[0] if cv_metrics else None), description="CV metric"),
    graph_key=widgets.Dropdown(options=sorted(graph_metrics),
                               value=(graph_metrics[0] if graph_metrics else None), description="Graph error"),
    show_error=widgets.Checkbox(value=False, description="Barres d’erreur"),
    alpha_pts=widgets.FloatSlider(value=0.9, min=0.2, max=1.0, step=0.1, description="Alpha points"),
    size_pts=widgets.IntSlider(value=50, min=10, max=120, step=5, description="Taille points"),
    show_labels=widgets.Checkbox(value=True, description="Étiquettes"),
    label_size=widgets.IntSlider(value=9, min=6, max=16, step=1, description="Taille texte"),
    show_tracks=widgets.Checkbox(value=True, description="Polylignes min→max"),
    with_arrows=widgets.Checkbox(value=True, description="Flèche direction"),
    track_lw=widgets.FloatSlider(value=1.8, min=0.5, max=4.0, step=0.1, description="Épaisseur ligne"),
    track_alpha=widgets.FloatSlider(value=0.75, min=0.2, max=1.0, step=0.05, description="Alpha ligne"),
)


In [ ]:
_df = DfCvGraph[['model','metric','abs_error_mean','std_abs_error', 'train_epoch_loss']].copy()
result = DfCvGraph[["model", "metric", "abs_error_mean"]].copy()

def _epoch_from_name(name: str) -> float:
    m = re.search(r'_(\d{2,4})(?:\D|$)', name)
    result = float(m.group(1)) if m else np.nan
    if result is np.nan:
        return -1 # TODO Peut être un problème - En pratique, pas vu
    return result


def _base_from_name(name: str) -> str:
    """(ex: Unet_bce_063 -> Unet_bce)"""
    i = name.rfind('_')
    return name[:i] if i >= 0 else name

_df['epoch'] = _df['model'].map(_epoch_from_name)
_df['base']  = _df['model'].map(_base_from_name)

_df

## Analyse avec des table de correlation et des regressions

In [ ]:
from sklearn.linear_model import TheilSenRegressor, LinearRegression
from scipy import stats

In [ ]:
def theil_sen(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)
    if len(x) < 2:
        return np.nan, np.nan
    model = TheilSenRegressor(max_iter=5000).fit(x, y)
    slope = float(model.coef_[0])
    intercept = float(model.intercept_)
    return slope, intercept

def linreg_ols(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)
    if len(x) < 2:
        return np.nan, np.nan, np.nan, np.nan
    model = LinearRegression().fit(x, y)
    slope = float(model.coef_[0])
    intercept = float(model.intercept_)
    r2 = float(model.score(x, y))

    # p-value
    n = len(y)
    y_hat = model.predict(x)
    residuals = y - y_hat
    s_err = np.sqrt(np.sum(residuals**2) / (n - 2))
    x_mean = x.mean()
    ssx = np.sum((x.flatten() - x_mean) ** 2)
    se_slope = s_err / np.sqrt(ssx)
    t_stat = slope / se_slope
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n-2))
    return slope, intercept, r2, p_value

# --- Spearman correlation (ρ + p-value) ---
def spearman_rho(x: np.ndarray, y: np.ndarray):
    rho, p_value = stats.spearmanr(x, y, nan_policy="omit")
    return float(rho), float(p_value)

def _lower_is_better(metric_name: str, y_col: str) -> bool:
    if y_col == 'abs_error_mean':
        return True
    m = metric_name.lower()
    return any(k in m for k in ['error', 'distance', 'loss', 'écart'])

def verdict_from_trend(metric_name: str, slope: float, rho: float, y_col='abs_error_mean', rho_eps: float = 0.35) -> str:
    if np.isnan(slope) or np.isnan(rho):
        return "Uncertain"
    lower_better = _lower_is_better(metric_name, y_col)
    if lower_better:
        if (slope < 0) and (rho <= -rho_eps): return "Improvement"
        if (slope > 0) and (rho >=  rho_eps): return "Deterioration"
    else:
        if (slope > 0) and (rho >=  rho_eps): return "Improvement"
        if (slope < 0) and (rho <= -rho_eps): return "Deterioration"
    return "Uncertain"


def compute_epoch_trends(df=_df, y_col='abs_error_mean') -> pd.DataFrame:
    rows = []
    for (base, metric), d in df.groupby(['base', 'metric']):
        d = d.dropna(subset=['epoch', y_col])
        if len(d) < 2:
            continue
        x = d['epoch'].values # d['epoch'].values train_epoch_loss
        y = d[y_col].values
        s_ts, b_ts = theil_sen(x, y)
        s_ols, b_ols, r2, p_val = linreg_ols(x, y)
        rho, p_val2 = spearman_rho(x, y)
        verdict = verdict_from_trend(metric, s_ts if np.isfinite(s_ts) else s_ols, rho, y_col=y_col)
        rows.append(dict(
            base=base, metric=metric, n_models=len(d),
            epoch_min=float(np.min(x)), epoch_max=float(np.max(x)),
            slope_theil=s_ts, slope_ols=s_ols, r2_ols=r2, spearman_rho=rho,
            verdict=verdict
        ))
    out = pd.DataFrame(rows).sort_values(['base','metric']).reset_index(drop=True)
    return out

TrendSummary = compute_epoch_trends()  # y_col='abs_error_mean' par défaut
TrendSummary

def filter_trends(verdict: str = "Improvement",
                  min_n: int = 2,
                  sort_by: str = "spearman_rho",
                  ascending: bool = False) -> pd.DataFrame:
    d = TrendSummary[TrendSummary['n_models'] >= min_n]
    d = d[d['verdict'] == verdict]
    if sort_by in d.columns:
        d = d.sort_values(sort_by, ascending=ascending)
    return d.reset_index(drop=True)

# %%
def plot_epoch_trend(base: str, metric: str, y_col: str = 'abs_error_mean', method: str = 'theil'):
    d = _df[(_df['base'] == base) & (_df['metric'] == metric)].dropna(subset=['epoch', y_col])
    if d.empty:
        print("Aucune donnée pour cette combinaison.")
        return
    d = d.sort_values('epoch')
    x = d['epoch'].values
    y = d[y_col].values

    if method == 'theil':
        slope, intercept = theil_sen(x, y)
        r2 = np.nan
    else:
        slope, intercept, r2, p_value = linreg_ols(x, y)

    xs = np.linspace(x.min(), x.max(), 100)
    ys = slope * xs + intercept

    plt.figure(figsize=(10, 6))
    plt.scatter(x, y)
    plt.plot(xs, ys, linewidth=2)

    rho, p_val = spearman_rho(x, y)
    verdict = verdict_from_trend(metric, slope, rho, y_col=y_col)
    print(r2)
    plt.title(f"{base} — {metric}  |  slope={slope:.4g},  ρ={rho:.3g}")
    plt.xlabel("Epoch")
    plt.ylabel(y_col)
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()

# Widgets
w_base   = widgets.Dropdown(options=sorted(_df['base'].unique().tolist()), description="Base")
w_metric = widgets.Dropdown(options=sorted(_df['metric'].unique().tolist()), description="Metric")
w_method = widgets.Dropdown(options=['theil','ols'], value='theil', description="Fit")
widgets.VBox([
    widgets.HBox([w_base, w_metric, w_method]),
    widgets.interactive_output(plot_epoch_trend, {'base': w_base, 'metric': w_metric, 'method': w_method})
])


In [ ]:
def theil_sen(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)
    if len(x) < 2:
        return np.nan, np.nan
    model = TheilSenRegressor(max_iter=5000).fit(x, y)
    slope = float(model.coef_[0])
    intercept = float(model.intercept_)
    return slope, intercept

def linreg_ols(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)
    if len(x) < 2:
        return np.nan, np.nan, np.nan, np.nan
    model = LinearRegression().fit(x, y)
    slope = float(model.coef_[0])
    intercept = float(model.intercept_)
    r2 = float(model.score(x, y))

    # p-value
    n = len(y)
    y_hat = model.predict(x)
    residuals = y - y_hat
    s_err = np.sqrt(np.sum(residuals**2) / (n - 2))
    x_mean = x.mean()
    ssx = np.sum((x.flatten() - x_mean) ** 2)
    se_slope = s_err / np.sqrt(ssx)
    t_stat = slope / se_slope
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n-2))
    return slope, intercept, r2, p_value

# --- Spearman correlation (ρ + p-value) ---
def spearman_rho(x: np.ndarray, y: np.ndarray):
    rho, p_value = stats.spearmanr(x, y, nan_policy="omit")
    return float(rho), float(p_value)

def _lower_is_better(metric_name: str, y_col: str) -> bool:
    if y_col == 'abs_error_mean':
        return True
    m = metric_name.lower()
    return any(k in m for k in ['error', 'distance', 'loss', 'écart'])

def verdict_from_trend(metric_name: str, slope: float, rho: float, y_col='abs_error_mean', rho_eps: float = 0.35) -> str:
    if np.isnan(slope) or np.isnan(rho):
        return "Uncertain"
    lower_better = _lower_is_better(metric_name, y_col)
    if lower_better:
        if (slope < 0) and (rho <= -rho_eps): return "Improvement"
        if (slope > 0) and (rho >=  rho_eps): return "Deterioration"
    else:
        if (slope > 0) and (rho >=  rho_eps): return "Improvement"
        if (slope < 0) and (rho <= -rho_eps): return "Deterioration"
    return "Uncertain"


def compute_epoch_trends(df=_df, y_col='abs_error_mean') -> pd.DataFrame:
    rows = []
    for (base, metric), d in df.groupby(['base', 'metric']):
        d = d.dropna(subset=['epoch', y_col])
        if len(d) < 2:
            continue
        x = d['epoch'].values # d['epoch'].values train_epoch_loss
        y = d[y_col].values
        s_ts, b_ts = theil_sen(x, y)
        s_ols, b_ols, r2, p_val = linreg_ols(x, y)
        rho, p_val2 = spearman_rho(x, y)
        verdict = verdict_from_trend(metric, s_ts if np.isfinite(s_ts) else s_ols, rho, y_col=y_col)
        rows.append(dict(
            base=base, metric=metric, n_models=len(d),
            epoch_min=float(np.min(x)), epoch_max=float(np.max(x)),
            slope_theil=s_ts, slope_ols=s_ols, r2_ols=r2, spearman_rho=rho,
            verdict=verdict
        ))
    out = pd.DataFrame(rows).sort_values(['base','metric']).reset_index(drop=True)
    return out

TrendSummary = compute_epoch_trends()  # y_col='abs_error_mean' par défaut
TrendSummary

def filter_trends(verdict: str = "Improvement",
                  min_n: int = 2,
                  sort_by: str = "spearman_rho",
                  ascending: bool = False) -> pd.DataFrame:
    d = TrendSummary[TrendSummary['n_models'] >= min_n]
    d = d[d['verdict'] == verdict]
    if sort_by in d.columns:
        d = d.sort_values(sort_by, ascending=ascending)
    return d.reset_index(drop=True)

# %%
def plot_epoch_trend(base: str, metric: str, y_col: str = 'abs_error_mean', method: str = 'theil'):
    d = _df[(_df['base'] == base) & (_df['metric'] == metric)].dropna(subset=['epoch', y_col])
    if d.empty:
        print("Aucune donnée pour cette combinaison.")
        return
    d = d.sort_values('epoch')
    x = d['epoch'].values
    y = d[y_col].values

    if method == 'theil':
        slope, intercept = theil_sen(x, y)
        r2 = np.nan
    else:
        slope, intercept, r2, p_value = linreg_ols(x, y)

    xs = np.linspace(x.min(), x.max(), 100)
    ys = slope * xs + intercept

    plt.figure(figsize=(10, 6))
    plt.scatter(x, y)
    plt.plot(xs, ys, linewidth=2)

    rho, p_val = spearman_rho(x, y)
    verdict = verdict_from_trend(metric, slope, rho, y_col=y_col)
    print(r2)
    plt.title(f"{base} — {metric}  |  slope={slope:.4g},  ρ={rho:.3g}")
    plt.xlabel("Epoch")
    plt.ylabel(y_col)
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()

# Widgets
w_base   = widgets.Dropdown(options=sorted(_df['base'].unique().tolist()), description="Base")
w_metric = widgets.Dropdown(options=sorted(_df['metric'].unique().tolist()), description="Metric")
w_method = widgets.Dropdown(options=['theil','ols'], value='theil', description="Fit")
widgets.VBox([
    widgets.HBox([w_base, w_metric, w_method]),
    widgets.interactive_output(plot_epoch_trend, {'base': w_base, 'metric': w_metric, 'method': w_method})
])


In [ ]:
from sklearn.linear_model import TheilSenRegressor, LinearRegression
from scipy import stats

def theil_sen(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)
    if len(x) < 2:
        return np.nan, np.nan
    model = TheilSenRegressor(max_iter=100000).fit(x, y)
    slope = float(model.coef_[0])
    intercept = float(model.intercept_)
    return slope, intercept

def linreg_ols(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)
    if len(x) < 2:
        return np.nan, np.nan, np.nan, np.nan
    model = LinearRegression().fit(x, y)
    slope = float(model.coef_[0])
    intercept = float(model.intercept_)
    r2 = float(model.score(x, y))

    # p-value
    n = len(y)
    y_hat = model.predict(x)
    residuals = y - y_hat
    s_err = np.sqrt(np.sum(residuals**2) / (n - 2))
    x_mean = x.mean()
    ssx = np.sum((x.flatten() - x_mean) ** 2)
    se_slope = s_err / np.sqrt(ssx)
    t_stat = slope / se_slope
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n-2))
    return slope, intercept, r2, p_value

# --- Spearman correlation (ρ + p-value) ---
def spearman_rho(x: np.ndarray, y: np.ndarray):
    rho, p_value = stats.spearmanr(x, y, nan_policy="omit")
    return float(rho), float(p_value)

def _lower_is_better(metric_name: str, y_col: str) -> bool:
    if y_col == 'abs_error_mean':
        return True
    m = metric_name.lower()
    return any(k in m for k in ['error', 'distance', 'loss', 'écart'])

def verdict_from_trend(metric_name: str, slope: float, rho: float, y_col='abs_error_mean', rho_eps: float = 0.35) -> str:
    if np.isnan(slope) or np.isnan(rho):
        return "Uncertain"
    lower_better = _lower_is_better(metric_name, y_col)
    if lower_better:
        if (slope < 0) and (rho <= -rho_eps): return "Improvement"
        if (slope > 0) and (rho >=  rho_eps): return "Deterioration"
    else:
        if (slope > 0) and (rho >=  rho_eps): return "Improvement"
        if (slope < 0) and (rho <= -rho_eps): return "Deterioration"
    return "Uncertain"


def compute_train_loss_trends(df=_df, y_col='abs_error_mean') -> pd.DataFrame:
    rows = []
    for (base, metric), d in df.groupby(['base', 'metric']):
        d = d.dropna(subset=['train_epoch_loss', y_col])
        if len(d) < 2:
            continue
        x = d['train_epoch_loss'].values # d['epoch'].values train_epoch_loss
        y = d[y_col].values
        s_ts, b_ts = theil_sen(x, y)
        s_ols, b_ols, r2, p_val = linreg_ols(x, y)
        rho, p_val2 = spearman_rho(x, y)
        verdict = verdict_from_trend(metric, s_ts if np.isfinite(s_ts) else s_ols, rho, y_col=y_col)
        rows.append(dict(
            base=base, metric=metric, n_models=len(d),
            train_epoch_loss_min=float(np.min(x)), train_epoch_loss_max=float(np.max(x)),
            slope_theil=s_ts, slope_ols=s_ols, r2_ols=r2, spearman_rho=rho,
            verdict=verdict
        ))
    out = pd.DataFrame(rows).sort_values(['base','metric']).reset_index(drop=True)
    return out

TrendSummary = compute_train_loss_trends()  # y_col='abs_error_mean' par défaut
TrendSummary

def filter_trends(verdict: str = "Improvement",
                  min_n: int = 2,
                  sort_by: str = "spearman_rho",
                  ascending: bool = False) -> pd.DataFrame:
    d = TrendSummary[TrendSummary['n_models'] >= min_n]
    d = d[d['verdict'] == verdict]
    if sort_by in d.columns:
        d = d.sort_values(sort_by, ascending=ascending)
    return d.reset_index(drop=True)

# %%
def plot_train_loss_trend(base: str, metric: str, y_col: str = 'abs_error_mean', method: str = 'theil'):
    d = _df[(_df['base'] == base) & (_df['metric'] == metric)].dropna(subset=['train_epoch_loss', y_col])
    if d.empty:
        print("Aucune donnée pour cette combinaison.")
        return
    d = d.sort_values('train_epoch_loss')
    x = d['train_epoch_loss'].values
    y = d[y_col].values

    if method == 'theil':
        slope, intercept = theil_sen(x, y)
        r2 = np.nan
    else:
        slope, intercept, r2, p_value = linreg_ols(x, y)

    xs = np.linspace(x.min(), x.max(), 100)
    ys = slope * xs + intercept

    plt.figure(figsize=(10, 6))
    plt.scatter(x, y)
    plt.plot(xs, ys, linewidth=2)

    rho, p_val = spearman_rho(x, y)
    verdict = verdict_from_trend(metric, slope, rho, y_col=y_col)
    plt.title(f"{base} — {metric}  |  slope={slope:.4g}, R²={r2 if np.isfinite(r2) else float('nan'):.3g},  ρ={rho:.3g}")
    plt.xlabel("Train loss")
    plt.ylabel(y_col)
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.savefig("train_loss_trend.png", transparent=True, dpi=600)
    plt.show()

# Widgets
w_base   = widgets.Dropdown(options=sorted(_df['base'].unique().tolist()), description="Base")
w_metric = widgets.Dropdown(options=sorted(_df['metric'].unique().tolist()), description="Metric")
w_method = widgets.Dropdown(options=['theil','ols'], value='theil', description="Fit")
widgets.VBox([
    widgets.HBox([w_base, w_metric, w_method]),
    widgets.interactive_output(plot_train_loss_trend, {'base': w_base, 'metric': w_metric, 'method': w_method})
])


In [ ]:
# ------------------------------------------------------------
# 2) Tableau R² (OLS) et ρ de Spearman pour chaque (base, metric)
# ------------------------------------------------------------
def r2_spearman_table(df: pd.DataFrame = None,
                      y_col: str = 'abs_error_mean') -> pd.DataFrame:
    """
    Calcule R² (OLS) et Spearman ρ pour chaque combinaison (base, metric)
    en utilisant train_epoch_loss comme variable explicative.
    """
    if df is None:
        df = _df
    rows = []
    for (base, metric), g in df.groupby(['base', 'metric']):
        gg = g.dropna(subset=['train_epoch_loss', y_col])
        if len(gg) < 2:
            continue
        x = gg['train_epoch_loss'].to_numpy().reshape(-1, 1)
        y = gg[y_col].to_numpy()

        # OLS
        _, _, r2, _ = linreg_ols(x, y)
        # Spearman
        rho, p_rho = spearman_rho(x.flatten(), y)

        rows.append({
            'base': base,
            'metric': metric,
            'n_points': int(len(gg)),
            'train_loss_min': float(np.min(x)),
            'train_loss_max': float(np.max(x)),
            'R2_OLS': float(r2) if np.isfinite(r2) else np.nan,
            'Spearman_rho': float(rho),
            'Spearman_pval': float(p_rho),
        })

    out = pd.DataFrame(rows).sort_values(['base', 'metric']).reset_index(drop=True)
    return out

# Construit puis affiche le tableau
R2_Spearman_Table = r2_spearman_table(_df, y_col='abs_error_mean')
R2_Spearman_Table

# Astuce : si vous voulez seulement (base, metric, R2, rho)
# display(R2_Spearman_Table[['base','metric','R2_OLS','Spearman_rho']])

In [ ]:
# --- Nouvelle clé "family" = premier token du base ---
# (base existe déjà via _base_from_name)
_df['family'] = _df['base'].map(lambda b: b.split('_')[0] if isinstance(b, str) else b)

# --- Version groupée par famille ---
def compute_epoch_trends_family(df=_df, y_col='abs_error_mean') -> pd.DataFrame:
    rows = []
    for (family, metric), d in df.groupby(['family', 'metric']):
        d = d.dropna(subset=['epoch', y_col])
        if len(d) < 2:
            continue
        x = d['epoch'].values
        y = d[y_col].values
        s_ts, b_ts = theil_sen(x, y)
        s_ols, b_ols, r2, p_val = linreg_ols(x, y)
        rho, p_val2 = spearman_rho(x, y)
        # verdict basé sur la même règle (lower is better, etc.)
        slope_for_verdict = s_ts if np.isfinite(s_ts) else s_ols
        verdict = verdict_from_trend(metric, slope_for_verdict, rho, y_col=y_col)
        rows.append(dict(
            family=family, metric=metric, n_models=len(d),
            epoch_min=float(np.min(x)), epoch_max=float(np.max(x)),
            slope_theil=s_ts, slope_ols=s_ols, r2_ols=r2, spearman_rho=rho,
            verdict=verdict
        ))
    out = pd.DataFrame(rows).sort_values(['family','metric']).reset_index(drop=True)
    return out

TrendSummaryFamily = compute_epoch_trends_family()  # y_col='abs_error_mean' par défaut
TrendSummaryFamily

def filter_trends_family(verdict: str = "Improvement",
                         min_n: int = 2,
                         sort_by: str = "spearman_rho",
                         ascending: bool = False) -> pd.DataFrame:
    d = TrendSummaryFamily[TrendSummaryFamily['n_models'] >= min_n]
    d = d[d['verdict'] == verdict]
    if sort_by in d.columns:
        d = d.sort_values(sort_by, ascending=ascending)
    return d.reset_index(drop=True)

import matplotlib.cm as cm
import matplotlib.colors as mcolors

def plot_epoch_trend_family(family: str, metric: str,
                            y_col: str = 'abs_error_mean',
                            method: str = 'theil'):
    d = _df[(_df['family'] == family) & (_df['metric'] == metric)].dropna(subset=['epoch', y_col])
    if d.empty:
        print("Aucune donnée pour cette combinaison.")
        return
    d = d.sort_values('epoch')

    # Couleur unique par base
    bases = sorted(d['base'].unique())
    cmap = mcolors.ListedColormap(BOLD_COLORS[:len(bases)])
    color_map = {b: cmap(i) for i, b in enumerate(bases)}

    x = d['epoch'].values
    y = d[y_col].values

    # Fit global (sur toute la famille)
    if method == 'theil':
        slope, intercept = theil_sen(x, y)
        r2 = np.nan
    else:
        slope, intercept, r2, p_value = linreg_ols(x, y)

    xs = np.linspace(x.min(), x.max(), 100)
    ys = slope * xs + intercept

    plt.figure(figsize=(10, 6))

    # Scatter par base avec couleur distincte
    for b in bases:
        db = d[d['base'] == b]
        plt.scatter(db['epoch'], db[y_col], label=b, color=color_map[b])

    # Ligne de tendance globale
    if method == "ols":
        method = "Ordinary Least Squares"
    plt.plot(xs, ys, 'k-', linewidth=2, label=f"{method} fit")

    rho, p_val = spearman_rho(x, y)
    verdict = verdict_from_trend(metric, slope, rho, y_col=y_col)

    plt.title(f"{family} — {metric}\n"
              f"slope={slope:.4g}, R²={r2 if np.isfinite(r2) else float('nan'):.3g}, "
              f"ρ={rho:.3g}")
    plt.xlabel("Epoch")
    plt.ylabel(y_col)
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


# --- Widgets version "family" ---
w_family = widgets.Dropdown(options=sorted(_df['family'].dropna().unique().tolist()), description="Family")
w_metric_f = widgets.Dropdown(options=sorted(_df['metric'].unique().tolist()), description="Metric")
w_method_f = widgets.Dropdown(options=['theil','ols'], value='theil', description="Fit")
widgets.VBox([
    widgets.HBox([w_family, w_metric_f, w_method_f]),
    widgets.interactive_output(plot_epoch_trend_family, {
        'family': w_family, 'metric': w_metric_f, 'method': w_method_f
    })
])


### Radar plot


In [ ]:
df_scatter

In [ ]:
df_radar = df_scatter.copy()

# normalisation entre 0 et 1 par métrique (min=0 + 0.1, max=1)
metrics = df_radar['metric'].unique()
for m in metrics:
    dm = df_radar[df_radar['metric'] == m]
    if dm.empty:
        continue
    min_val = dm['abs_error_mean'].min()
    max_val = dm['abs_error_mean'].max()
    range_val = max_val - min_val
    if range_val == 0:
        df_radar.loc[df_radar['metric'] == m, 'norm_error'] = 0.5  # valeur arbitraire si pas de variation
    else:
        df_radar.loc[df_radar['metric'] == m, 'norm_error'] = 0.1 + 0.9 * (dm['abs_error_mean'] - min_val) / range_val
df_radar

In [ ]:
# Axes
graph_metrics_all = sorted(df_radar["metric"].dropna().unique().tolist())
cv_metrics_all = sorted(cv_metrics_results.keys())

print("Graph metrics:", graph_metrics_all)
print("CV metrics:", cv_metrics_all)

In [ ]:

def _radar_angles(n):
    ang = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    return ang + ang[:1]


def _plot_poly(ax, angles, values, label, color, fill=False, alpha=0.18, marker=None):
    vals = list(values) + [values[0]]
    ax.plot(angles, vals, linewidth=2, color=color, label=label, marker=marker)
    if fill:
        ax.fill(angles, vals, alpha=alpha, color=color)


def _setup_ax(ax, labels, title):
    angles = _radar_angles(len(labels))
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_title(title, pad=12, fontsize=12)
    return angles


from sklearn.preprocessing import MinMaxScaler
import numpy as np

def _scale_values_per_axis(values_by_model: dict):

    arr = np.array(list(values_by_model.values()), dtype=float)
    mask = np.isnan(arr)
    col_means = np.nanmean(arr, axis=0)
    arr[mask] = np.take(col_means, np.where(mask)[1])

    scaler = MinMaxScaler(feature_range=(0, 1))
    arr_scaled = scaler.fit_transform(arr)

    out = {}
    for i, m in enumerate(values_by_model.keys()):
        out[m] = arr_scaled[i, :].tolist()
    return out


def plot_double_radar(models_to_show, graph_axes, cv_axes, fill_polys=True, show_legend=True):
    # Garde un ordre stable
    models = [m for m in models_all if m in set(models_to_show)]
    if len(models) == 0:
        print("Sélectionne au moins un modèle.")
        return

    graph_axes = [m for m in graph_metrics_all if m in set(graph_axes)]
    cv_axes = [m for m in cv_metrics_all if m in set(cv_axes)]
    if len(graph_axes) < 3 or len(cv_axes) < 3:
        print("Il faut au moins 3 axes par radar.")
        return

    # Prépare les valeurs par modèle
    # Radar gauche: pour chaque metric interne -> norm_abs_approx moyen (déjà en 0..1, plus haut = mieux)
    left_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m][['metric', 'abs_error_mean']]
        mp = dict(zip(d['metric'], d['abs_error_mean']))
        left_vals_by_model[m] = [float(mp.get(ax, np.nan)) for ax in graph_axes]

    # Radar droit: métriques CV -> colonnes (scores 0..1, les 'error' déjà inversées plus haut)
    right_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m].iloc[:1]  # valeurs CV identiques sur les lignes du modèle
        if d.empty:
            right_vals_by_model[m] = [np.nan] * len(cv_axes)
        else:
            row = d.iloc[0]
            right_vals_by_model[m] = [float(row[ax]) for ax in cv_axes]
    
    left_vals_by_model = _scale_values_per_axis(left_vals_by_model)
    right_vals_by_model = _scale_values_per_axis(right_vals_by_model)
    
    # Figure
    _ = plt.figure(figsize=(18, 10))
    ax_left = plt.subplot(1, 2, 1, projection='polar')
    ax_right = plt.subplot(1, 2, 2, projection='polar')

    # Axes / titres
    ang_left = _setup_ax(ax_left, graph_axes, "Metrics from graph evaluation")
    ang_right = _setup_ax(ax_right, cv_axes, "Metrics from segmentation evaluation")
    
    # invert graph axes (norm_abs: plus bas = mieux)
    ax_left.invert_yaxis()

    # Un polygone par modèle sur chaque radar
    markers = ['o', 's', 'D', '^', 'v', 'X', 'P', '*']
    for i, m in enumerate(models):
        col = color_map[m]
        mrk = markers[i % len(markers)]
        # gauche
        lv = left_vals_by_model[m]
        _plot_poly(ax_left, ang_left, lv, label=m, color=col, fill=fill_polys, marker=mrk)
        # droite
        rv = right_vals_by_model[m]
        _plot_poly(ax_right, ang_right, rv, label=m, color=col, fill=fill_polys, marker=mrk)

    # Légende commune
    if show_legend:
        handles = [plt.Line2D([0], [0], marker=markers[i % len(markers)], color='w',
                              markerfacecolor=color_map[m], markersize=8, label=m)
                   for i, m in enumerate(models)]
        ax_right.legend(handles=handles, title="Modèles",
                        bbox_to_anchor=(1.25, 1.0), loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()


# --- Widgets ---
w_models = widgets.SelectMultiple(options=models_all, value=tuple(models_all), description="Modèles",
                                  rows=min(8, len(models_all)))
w_graph = widgets.SelectMultiple(options=graph_metrics_all, value=tuple(graph_metrics_all), description="Graph axes",
                                 rows=min(10, len(graph_metrics_all)))
w_cv = widgets.SelectMultiple(options=cv_metrics_all, value=tuple(cv_metrics_all), description="CV axes",
                              rows=len(cv_metrics_all))
w_fill = widgets.Checkbox(value=True, description="Remplir")
w_legend = widgets.Checkbox(value=True, description="Légende")

widgets.interactive(
    plot_double_radar,
    models_to_show=w_models,
    graph_axes=w_graph,
    cv_axes=w_cv,
    fill_polys=w_fill,
    show_legend=w_legend
)

###### Ordered

In [ ]:
# ---------- Optimisation d'ordre d'axes pour radars ----------
import numpy as np

def _nan_corr(a: np.ndarray, b: np.ndarray) -> float:
    """Corrélation de Pearson avec gestion des NaN."""
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 2:
        return 0.0
    return float(np.corrcoef(a[mask], b[mask])[0, 1])

def _axes_distance_matrix(values_by_model: dict[str, list[float]],
                          invert_low_is_better: bool = True,
                          lower_is_better_names: list[str] | None = None,
                          axis_names: list[str] | None = None) -> tuple[np.ndarray, list[str]]:
    """
    values_by_model: dict[model] -> list of axis values (dans l'ordre fourni par axis_names)
    Retourne D (distances 1-corr) et la liste alignée des axes.
    """
    # Déduire les noms d'axes
    if axis_names is None:
        # Tous les modèles doivent avoir la même longueur
        n_axes = len(next(iter(values_by_model.values())))
        axis_names = [f"axis_{i}" for i in range(n_axes)]
    n_axes = len(axis_names)

    # Matrice M (n_models x n_axes)
    Ms = []
    for m, vals in values_by_model.items():
        Ms.append(np.asarray(vals, float))
    M = np.vstack(Ms)  # (n_models, n_axes)

    # Option: inverser les axes "lower is better" pour que "plus haut = mieux"
    if invert_low_is_better and lower_is_better_names is not None:
        name_to_idx = {n: i for i, n in enumerate(axis_names)}
        for n in lower_is_better_names:
            if n in name_to_idx:
                j = name_to_idx[n]
                col = M[:, j]
                # min-max par colonne avec NaN
                cmin = np.nanmin(col); cmax = np.nanmax(col)
                if np.isfinite(cmin) and np.isfinite(cmax) and cmax > cmin:
                    M[:, j] = (cmax - col) / (cmax - cmin)  # inversion normalisée
                else:
                    # fallback: inversion simple
                    M[:, j] = -col

    # Remplacer NaN par moyenne de colonne (évite corr NaN)
    col_means = np.nanmean(M, axis=0)
    inds = np.where(~np.isfinite(M))
    M[inds] = np.take(col_means, inds[1])

    # Distance = 1 - corr entre axes (corr sur colonnes)
    D = np.zeros((n_axes, n_axes), float)
    for i in range(n_axes):
        for j in range(i+1, n_axes):
            r = _nan_corr(M[:, i], M[:, j])
            Dij = 1.0 - r
            D[i, j] = D[j, i] = max(0.0, min(2.0, Dij))  # borne [0,2]
    return D, axis_names

def _tsp_greedy_cycle(D: np.ndarray) -> list[int]:
    """Heuristique gloutonne pour un cycle: démarre au meilleur pair, étend par plus proche voisin."""
    n = D.shape[0]
    if n <= 3:
        return list(range(n))
    # meilleur pair initial (distance minimale)
    i0, j0 = np.unravel_index(np.argmin(np.where(np.eye(n, dtype=bool), np.inf, D)), D.shape)
    path = [i0, j0]
    remaining = set(range(n)) - set(path)
    # on construit un chemin (pas encore cycle) en ajoutant au mieux à une des extrémités
    while remaining:
        # coût pour ajouter k à gauche (path[0]) ou à droite (path[-1])
        best_k, best_side, best_cost = None, None, np.inf
        left, right = path[0], path[-1]
        for k in list(remaining):
            c_left = D[k, left]
            c_right = D[right, k]
            if c_left < best_cost:
                best_k, best_side, best_cost = k, 'left', c_left
            if c_right < best_cost:
                best_k, best_side, best_cost = k, 'right', c_right
        if best_side == 'left':
            path.insert(0, best_k)
        else:
            path.append(best_k)
        remaining.remove(best_k)
    # convertit en cycle en choisissant la rotation qui minimise le coût cyclique
    # (toute rotation est équivalente visuellement sur un radar)
    return path

def _two_opt_cycle(order: list[int], D: np.ndarray, max_iter: int = 200) -> list[int]:
    """Amélioration locale 2-opt sur cycle."""
    n = len(order)
    if n < 4:
        return order
    improved = True
    it = 0
    def cycle_cost(ordr):
        return sum(D[ordr[i], ordr[(i+1) % n]] for i in range(n))
    best = order[:]
    best_cost = cycle_cost(best)
    while improved and it < max_iter:
        improved = False
        it += 1
        for i in range(n - 1):
            for k in range(i + 2, n - (0 if i > 0 else 1)):
                new = best[:i+1] + best[i+1:k+1][::-1] + best[k+1:]
                c = sum(D[new[t], new[(t+1) % n]] for t in range(n))
                if c + 1e-9 < best_cost:
                    best, best_cost, improved = new, c, True
        if not improved:
            break
    return best

def compute_optimal_axis_order(values_by_model: dict[str, list[float]],
                               axis_names: list[str],
                               lower_is_better_names: list[str] | None = None) -> list[str]:
    """
    Calcule un ordre circulaire d'axes qui met côte-à-côte des axes corrélés
    (distance = 1 - corr), pour lisser les polygones (tous modèles confondus).
    """
    D, names = _axes_distance_matrix(
        values_by_model,
        invert_low_is_better=True,
        lower_is_better_names=lower_is_better_names,
        axis_names=axis_names
    )
    init = _tsp_greedy_cycle(D)
    improved = _two_opt_cycle(init, D, max_iter=200)
    return [names[i] for i in improved]


def plot_double_radar(models_to_show, graph_axes, cv_axes, fill_polys=True, show_legend=True,
                      optimize_order: bool = True):
    # ... (même en-tête que ton code)
    models = [m for m in models_all if m in set(models_to_show)]
    # validations identiques…

    # ----------------- Préparation des valeurs (non-scalées d'abord) -----------------
    # GRAPH: abs_error_mean (plus bas = mieux) -> on indiquera ces noms au solveur pour inversion
    left_vals_by_model_raw = {}
    for m in models:
        d = df_radar[df_radar['model'] == m][['metric', 'abs_error_mean']]
        mp = dict(zip(d['metric'], d['abs_error_mean']))
        left_vals_by_model_raw[m] = [float(mp.get(ax, np.nan)) for ax in graph_axes]

    # CV: scores 0..1 (déjà dans le bon sens chez toi)
    right_vals_by_model_raw = {}
    for m in models:
        d = df_radar[df_radar['model'] == m].iloc[:1]
        if d.empty:
            right_vals_by_model_raw[m] = [np.nan] * len(cv_axes)
        else:
            row = d.iloc[0]
            right_vals_by_model_raw[m] = [float(row[ax]) for ax in cv_axes]

    # ----------------- Optimisation d'ordre d'axes -----------------
    if optimize_order and len(graph_axes) >= 3:
        # toutes les axes graph sont "lower is better" ici (abs_error_mean par axe)
        graph_axes = compute_optimal_axis_order(
            left_vals_by_model_raw, graph_axes,
            lower_is_better_names=graph_axes  # on inverse toutes pour la corrélation
        )
    if optimize_order and len(cv_axes) >= 3:
        # côté CV: si certaines sont "lower is better", liste-les ici, p.ex.:
        cv_lower = [ax for ax in cv_axes if any(k in ax.lower() for k in ['error','distance','loss','écart'])]
        cv_axes = compute_optimal_axis_order(
            right_vals_by_model_raw, cv_axes,
            lower_is_better_names=cv_lower
        )

    # ----------------- Passage à ton flux existant (scaling, plot, etc.) -----------------
    # Reconstruire avec axes ordonnés
    left_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m][['metric', 'abs_error_mean']]
        mp = dict(zip(d['metric'], d['abs_error_mean']))
        left_vals_by_model[m] = [float(mp.get(ax, np.nan)) for ax in graph_axes]

    right_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m].iloc[:1]
        if d.empty:
            right_vals_by_model[m] = [np.nan] * len(cv_axes)
        else:
            row = d.iloc[0]
            right_vals_by_model[m] = [float(row[ax]) for ax in cv_axes]

    # Normalisation colonne-à-colonne (comme tu fais)
    left_vals_by_model = _scale_values_per_axis(left_vals_by_model)
    right_vals_by_model = _scale_values_per_axis(right_vals_by_model)

    # --- le reste de ton code inchangé : création figure, _setup_ax, _plot_poly, etc. ---
    _ = plt.figure(figsize=(18, 10))
    ax_left = plt.subplot(1, 2, 1, projection='polar')
    ax_right = plt.subplot(1, 2, 2, projection='polar')

    ang_left = _setup_ax(ax_left, graph_axes, "Metrics from graph evaluation")
    ang_right = _setup_ax(ax_right, cv_axes, "Metrics from segmentation evaluation")
    ax_left.invert_yaxis()

    markers = ['o', 's', 'D', '^', 'v', 'X', 'P', '*']
    for i, m in enumerate(models):
        col = color_map[m]
        mrk = markers[i % len(markers)]
        _plot_poly(ax_left,  ang_left,  left_vals_by_model[m],  label=m, color=col, fill=fill_polys, marker=mrk)
        _plot_poly(ax_right, ang_right, right_vals_by_model[m], label=m, color=col, fill=fill_polys, marker=mrk)

    if show_legend:
        handles = [plt.Line2D([0], [0], marker=markers[i % len(markers)], color='w',
                              markerfacecolor=color_map[m], markersize=8, label=m)
                   for i, m in enumerate(models)]
        ax_right.legend(handles=handles, title="Modèles",
                        bbox_to_anchor=(1.25, 1.0), loc='upper left', fontsize=9)
    plt.savefig("radar_comparison.png", transparent=True, dpi=600)
    plt.savefig("radar_comparison_with_legend.png", transparent=True, dpi=600, bbox_inches='tight')
    plt.tight_layout()
    plt.show()


In [ ]:
w_opt = widgets.Checkbox(value=True, description="Optimiser l’ordre des axes")

widgets.interactive(
    lambda models_to_show, graph_axes, cv_axes, fill_polys, show_legend, optimize_order:
        plot_double_radar(models_to_show, graph_axes, cv_axes, fill_polys, show_legend, optimize_order),
    models_to_show=w_models,
    graph_axes=w_graph,
    cv_axes=w_cv,
    fill_polys=w_fill,
    show_legend=w_legend,
    optimize_order=w_opt
)


#### Corr matrix

In [ ]:
# %% [markdown]
# ### CV ➜ Graph: matrice de corrélation (Spearman)

from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Prépare colonnes CV & colonnes graph ---
cv_cols = [c for c in DfCvGraph.columns if c not in ["model","metric","abs_error_mean","std_abs_error"]]
cv_cols = ["Precision", "Recall", "F1Score", "MeanIoU", "HausdorffDistance", "APLS", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]
graph_metrics = sorted(DfCvGraph["metric"].dropna().unique().tolist())
graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule ρ et p-valeurs par (cv, graph_metric) ---
rho_rows, p_rows = [], []
for cv in cv_cols:
    rho_row, p_row = {}, {}
    for g in graph_metrics:
        d = DfCvGraph[DfCvGraph["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            d_cv = d[cv]
            if cv in ["HausdorffDistance", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]:
                d_cv = -d_cv # Parce que distance donc inverser les score
            rho, p = spearmanr(d_cv, -d["abs_error_mean"])
        rho_row[g] = float(rho) if np.isfinite(rho) else np.nan
        p_row[g]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)


print("Corrélation (ρ Spearman):")
display(CorrSpearman.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

print("Significativité (*<.05, **<.01, ***<.001):")

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.45*len(cv_cols)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Spearman ρ (CV vs abs_error_mean)")
plt.yticks(range(len(cv_cols)), cv_cols)
plt.xticks(range(len(graph_metrics)), graph_metrics, rotation=45, ha="right")
plt.title("Correlation table (Spearman) between Segmentation metrics and Graph error", pad=20)
# Annotations: ρ + étoiles
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        rho = CorrSpearman.iat[i, j]
        p = Pvalues.iat[i, j]
        if np.isnan(p):
            s = ""
        elif p < 0.001:
            s = "***"
        elif p < 0.01:
            s = "**"
        elif p < 0.05:
            s = "*"
        else:
            s = ""
        s = ""
        txt = "" if np.isnan(rho) else f"{rho:.4f}{s}"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_cv_to_graph_abs_error_spearman.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrSpearman.to_csv("corr_cv_to_graph_abs_error_spearman.csv")
#Pvalues.to_csv("corr_cv_to_graph_abs_error_pvalues.csv")


In [ ]:
# --- Improved, poster-ready Spearman heatmap: Segmentation metrics vs Graph error ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# =========================
# 0) Inputs / configuration
# =========================
# Colonnes CV à garder (ordre d'affichage)
cv_cols = [
    "Precision", "Recall", "F1Score", "MeanIoU",
    "HausdorffDistance", "APLS", "AverageCenterlineDistance",
    "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"
]

# Métriques pour lesquelles une PLUS PETITE valeur = MEILLEUR (on inverse le signe avant corrélation)
distance_like = {"HausdorffDistance", "AverageCenterlineDistance",
                 "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"}

# Mapping jolis noms (axes et titre poster)
cv_pretty_map = {
    "Precision": "Precision",
    "Recall": "Recall",
    "F1Score": "F1 Score",
    "MeanIoU": "Mean IoU",
    "HausdorffDistance": "Hausdorff Distance",
    "APLS": "APLS",
    "AverageCenterlineDistance": "Average Centerline Distance",
    "Betti0VariationIndexGPU": "β0 Variation Index",
    "Betti1VariationIndexGPU": "β1 Variation Index",
}

def pretty_metric(name: str) -> str:
    # Ajuste ici si tu veux des libellés plus humains
    repl = {
        "TotalRootLength": "Total Root Length",
        "PrimaryRootLength": "Primary Root Length",
        "LateralRootLength": "Lateral Root Length",
        "NumberOfOrgans": "Number of Organs",
        "RootDensity": "Root Density",
        "ConvexHullArea": "Convex Hull Area",
        "ConvexHullVolume": "Convex Hull Volume",
        "NumberOfLateralRoots": "Number of Lateral Roots",
    }
    if name in repl:
        return repl[name]
    return name.replace("_", " ").replace("-", " ").title()

# =========================
# 1) Prépare lignes/colonnes
# =========================
graph_metrics = sorted(DfCvGraph["metric"].dropna().unique().tolist())
if "NumberOfLateralRoots" in graph_metrics:
    graph_metrics.remove("NumberOfLateralRoots")

# Filtre les colonnes manquantes si besoin (robuste)
cv_cols = [c for c in cv_cols if c in DfCvGraph.columns]

# =========================
# 2) Corrélations de Spearman
#    Convention: corr élevée = meilleur alignement (↓erreur graph)
# =========================
rho_rows, p_rows = [], []
for cv in cv_cols:
    rho_row, p_row = {}, {}
    for g in graph_metrics:
        d = DfCvGraph[DfCvGraph["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            # pour les "distances", on inverse pour que "plus haut = mieux"
            d_cv = -d[cv] if cv in distance_like else d[cv]
            # On corrèle avec le signe négatif de l'erreur (plus petite = mieux)
            rho, p = spearmanr(d_cv, -d["abs_error_mean"])
        rho_row[g] = float(rho) if np.isfinite(rho) else np.nan
        p_row[g]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)

# (Option) Aperçu console
print("Spearman Correlation Coefficients (ρ):")
display(CorrSpearman.round(3))
print("Associated p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))
print("Significance Levels (*p* < .05, **p* < .01, ***p* < .001)")

# =========================
# 3) Affichage "poster-ready"
# =========================
# Labels propres
y_labels = [cv_pretty_map.get(c, c) for c in cv_cols]
x_labels = [pretty_metric(g) for g in graph_metrics]

# Masque NaN pour un rendu propre
M = CorrSpearman.values
mask = ~np.isfinite(M)
M_masked = np.ma.masked_where(mask, M)

plt.figure(figsize=(1.25*len(graph_metrics)+3, 0.5*len(cv_cols)+3))

# Heatmap centrée [-1,1] + couleur des NaN
cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad(color="#E6E6E6")  # gris clair pour NaN
im = plt.imshow(M_masked, aspect="auto", vmin=-1, vmax=1, cmap=cmap, interpolation="nearest")

# Colorbar
cbar = plt.colorbar(im, fraction=0.046, pad=0.05)
cbar.set_label("Spearman’s ρ (Segmentation Metric vs. Graph Error)")

# Axes
plt.yticks(range(len(cv_cols)), y_labels)
plt.xticks(range(len(graph_metrics)), x_labels, rotation=45, ha="right")

# Titre
plt.title("Segmentation Metrics vs. Graph-Level Error — Spearman Correlation", pad=20)

# Quadrillage léger pour lecture
plt.gca().set_xticks(np.arange(-0.5, len(graph_metrics), 1), minor=True)
plt.gca().set_yticks(np.arange(-0.5, len(cv_cols), 1), minor=True)
plt.grid(which="minor", color="white", linewidth=0.8, alpha=0.6)
plt.gca().tick_params(axis='both', which='both', length=0)

# --- Annotate cells: ρ + étoiles (contraste auto) ---
def stars_from_p(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001: return ""
    if p < 0.01:  return ""
    if p < 0.05:  return ""
    return ""

for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        rho = CorrSpearman.iat[i, j]
        p   = Pvalues.iat[i, j]
        if not np.isfinite(rho):
            continue
        s   = stars_from_p(p)
        txt = f"{rho:.3f}{s}"
        # Contraste: texte blanc si |rho| > 0.5
        color = "black" if abs(rho) >= 0.5 else "black"
        plt.text(j, i, txt, ha="center", va="center", fontsize=8.5, color=color)

plt.tight_layout()
plt.savefig("corr_segmetrics_to_graph_abs_error_spearman.png", transparent=True, dpi=600)
plt.show()

# (Option) export CSV
# CorrSpearman.to_csv("corr_segmetrics_to_graph_abs_error_spearman.csv")
# Pvalues.to_csv("corr_segmetrics_to_graph_abs_error_pvalues.csv")


In [ ]:
# --- Heatmap: each cell colored by (R²+ρ)/2, text shows R² | Spearman ρ ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LinearRegression

# =========================
# 0) Inputs / configuration
# =========================
cv_cols = [
    "Precision", "Recall", "F1Score", "MeanIoU",
    "HausdorffDistance", "AverageCenterlineDistance",
    "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"
]

# Métriques pour lesquelles une PLUS PETITE valeur = MEILLEUR (on inverse avant corrélation / régression)
distance_like = {"HausdorffDistance", "AverageCenterlineDistance",
                 "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"}

cv_pretty_map = {
    "Precision": "Precision",
    "Recall": "Recall",
    "F1Score": "F1 Score",
    "MeanIoU": "Mean IoU",
    "HausdorffDistance": "Hausdorff Distance",
    "APLS": "APLS",
    "AverageCenterlineDistance": "Average Centerline Distance",
    "Betti0VariationIndexGPU": "β0 Variation Index",
    "Betti1VariationIndexGPU": "β1 Variation Index",
}

def pretty_metric(name: str) -> str:
    repl = {
        "TotalRootLength": "Total Root Length",
        "PrimaryRootLength": "Primary Root Length",
        "LateralRootLength": "Lateral Root Length",
        "NumberOfOrgans": "Number of Organs",
        "RootDensity": "Root Density",
        "ConvexHullArea": "Convex Hull Area",
        "ConvexHullVolume": "Convex Hull Volume",
        "NumberOfLateralRoots": "Number of Lateral Roots",
    }
    if name in repl:
        return repl[name]
    return name.replace("_", " ").replace("-", " ").title()

# =========================
# 1) Prépare lignes/colonnes
# =========================
graph_metrics = sorted(DfCvGraph["metric"].dropna().unique().tolist())
if "NumberOfLateralRoots" in graph_metrics:
    graph_metrics.remove("NumberOfLateralRoots")

cv_cols = [c for c in cv_cols if c in DfCvGraph.columns]  # robustesse

# =========================
# 2) Calcul de R² (Pearson/OLS) et de Spearman ρ
#    Convention: on travaille avec d_cv et -abs_error_mean pour "plus haut = mieux"
# =========================
R2   = pd.DataFrame(index=cv_cols, columns=graph_metrics, dtype=float)
Rho  = pd.DataFrame(index=cv_cols, columns=graph_metrics, dtype=float)

for cv in cv_cols:
    for g in graph_metrics:
        d = DfCvGraph[DfCvGraph["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            R2.loc[cv, g]  = np.nan
            Rho.loc[cv, g] = np.nan
            continue
        # Inversion pour rendre "plus haut = mieux" cohérent
        d_cv = -d[cv] if cv in distance_like else d[cv]
        y = -d["abs_error_mean"]

        # Pearson r -> R²
        r, _ = pearsonr(d_cv.values, y.values)
        R2.loc[cv, g] = float(r**2)

        # Spearman ρ
        rho, _ = spearmanr(d_cv.values, y.values)
        Rho.loc[cv, g] = float(rho) if np.isfinite(rho) else np.nan

# =========================
# 3) Affichage "poster-ready"
# =========================
y_labels = [cv_pretty_map.get(c, c) for c in cv_cols]
x_labels = [pretty_metric(g) for g in graph_metrics]

# Couleur des cellules = mix (R² + ρ) / 2  (comme dans ton autre plot)
color_metric = (Rho + R2) / 2.0
M = color_metric.values.astype(float)

# Masque NaN et colormap
mask = ~np.isfinite(M)
M_masked = np.ma.masked_where(mask, M)

plt.figure(figsize=(1.25*len(graph_metrics)+3, 0.5*len(cv_cols)+3))

cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad(color="#E6E6E6")  # gris clair pour NaN
im = plt.imshow(M_masked, aspect="auto", vmin=-1, vmax=1, cmap=cmap, interpolation="nearest")

# Colorbar
cbar = plt.colorbar(im, fraction=0.046, pad=0.05)

# Axes
plt.yticks(range(len(cv_cols)), y_labels)
plt.xticks(range(len(graph_metrics)), x_labels, rotation=45, ha="right")

plt.title("Segmentation Metrics vs. Graph-Level Error — R² | Spearman ρ (Cell color = (R²+ρ)/2)", pad=20)

# Quadrillage léger
ax = plt.gca()
ax.set_xticks(np.arange(-0.5, len(graph_metrics), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(cv_cols), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=0.8, alpha=0.6)
ax.tick_params(axis='both', which='both', length=0)

# --- Texte dans chaque cellule: R² | ρ, avec contraste auto (blanc si |mix| ≥ 0.8) ---
for i, cv in enumerate(cv_cols):
    for j, g in enumerate(graph_metrics):
        r2  = R2.loc[cv, g]
        rho = Rho.loc[cv, g]
        if not (np.isfinite(r2) and np.isfinite(rho)):
            continue
        txt = f"R²={r2:.2f}\nρ={rho:.2f}"
        colval = color_metric.loc[cv, g]
        color = "white" if (np.isfinite(colval) and abs(colval) >= 0.80) else "black"
        plt.text(j, i, txt, ha="center", va="center", fontsize=8.5, color=color)

plt.tight_layout()
plt.savefig("segmetrics_vs_graph_error_heatmap_r2_rho_mixedcolor.png", transparent=True, dpi=600)
plt.show()

# (Option) exports CSV
# R2.to_csv("segmetrics_vs_graph_error_R2.csv")
# Rho.to_csv("segmetrics_vs_graph_error_Spearman.csv")
# color_metric.to_csv("segmetrics_vs_graph_error_color_metric_mix.csv")


In [ ]:
# %% [markdown]
# ### CV ➜ Graph: matrice de corrélation (Spearman)

from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DfCvGraph2 = DfCvGraph[DfCvGraph["model"].str.contains("Unet")].copy()
# --- Prépare colonnes CV & colonnes graph ---
cv_cols = [c for c in DfCvGraph2.columns if c not in ["model","metric","abs_error_mean","std_abs_error"]]
cv_cols = ["Precision", "Recall", "F1Score", "MeanIoU", "HausdorffDistance", "APLS", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]
graph_metrics = sorted(DfCvGraph2["metric"].dropna().unique().tolist())
graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule ρ et p-valeurs par (cv, graph_metric) ---
rho_rows, p_rows = [], []
for cv in cv_cols:
    rho_row, p_row = {}, {}
    for g in graph_metrics:
        d = DfCvGraph2[DfCvGraph2["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            d_cv = d[cv]
            if cv in ["HausdorffDistance", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]:
                d_cv = -d_cv # Parce que distance donc inverser les score
            rho, p = spearmanr(d_cv, -d["abs_error_mean"])
        rho_row[g] = float(rho) if np.isfinite(rho) else np.nan
        p_row[g]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)


print("Corrélation (ρ Spearman):")
display(CorrSpearman.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

print("Significativité (*<.05, **<.01, ***<.001):")

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.45*len(cv_cols)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Spearman ρ (CV vs abs_error_mean)")
plt.yticks(range(len(cv_cols)), cv_cols)
plt.xticks(range(len(graph_metrics)), graph_metrics, rotation=45, ha="right")
plt.title("Unet only Correlation table (Spearman) between Segmentation metrics and Graph error", pad=20)
# Annotations: ρ + étoiles
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        rho = CorrSpearman.iat[i, j]
        p = Pvalues.iat[i, j]
        if np.isnan(p):
            s = ""
        elif p < 0.001:
            s = "***"
        elif p < 0.01:
            s = "**"
        elif p < 0.05:
            s = "*"
        else:
            s = ""
        s = ""
        txt = "" if np.isnan(rho) else f"{rho:.4f}{s}"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_cv_to_graph_abs_error_spearman.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrSpearman.to_csv("corr_cv_to_graph_abs_error_spearman.csv")
#Pvalues.to_csv("corr_cv_to_graph_abs_error_pvalues.csv")


In [ ]:
# %% [markdown]
# ### CV ➜ Graph: matrice de corrélation (Spearman)

from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DfCvGraph2 = DfCvGraph[DfCvGraph["model"].str.contains("Segformer")].copy()
print(f"Modèles Segformer: {DfCvGraph2['model'].nunique()} sur {DfCvGraph['model'].nunique()} modèles au total")
# --- Prépare colonnes CV & colonnes graph ---
cv_cols = [c for c in DfCvGraph2.columns if c not in ["model","metric","abs_error_mean","std_abs_error"]]
cv_cols = ["Precision", "Recall", "F1Score", "MeanIoU", "HausdorffDistance", "APLS", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]
graph_metrics = sorted(DfCvGraph2["metric"].dropna().unique().tolist())
graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule ρ et p-valeurs par (cv, graph_metric) ---
rho_rows, p_rows = [], []
for cv in cv_cols:
    rho_row, p_row = {}, {}
    for g in graph_metrics:
        d = DfCvGraph2[DfCvGraph2["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            d_cv = d[cv]
            if cv in ["HausdorffDistance", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]:
                d_cv = -d_cv # Parce que distance donc inverser les score
            rho, p = spearmanr(d_cv, -d["abs_error_mean"])
        rho_row[g] = float(rho) if np.isfinite(rho) else np.nan
        p_row[g]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)


print("Corrélation (ρ Spearman):")
display(CorrSpearman.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

print("Significativité (*<.05, **<.01, ***<.001):")

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.45*len(cv_cols)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Spearman ρ (CV vs abs_error_mean)")
plt.yticks(range(len(cv_cols)), cv_cols)
plt.xticks(range(len(graph_metrics)), graph_metrics, rotation=45, ha="right")
plt.title("Segformer only - Correlation table (Spearman) between Segmentation metrics and Graph error", pad=20)
# Annotations: ρ + étoiles
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        rho = CorrSpearman.iat[i, j]
        p = Pvalues.iat[i, j]
        if np.isnan(p):
            s = ""
        elif p < 0.001:
            s = "***"
        elif p < 0.01:
            s = "**"
        elif p < 0.05:
            s = "*"
        else:
            s = ""
        s = ""
        txt = "" if np.isnan(rho) else f"{rho:.4f}{s}"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_cv_to_graph_abs_error_spearman.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrSpearman.to_csv("corr_cv_to_graph_abs_error_spearman.csv")
#Pvalues.to_csv("corr_cv_to_graph_abs_error_pvalues.csv")


In [ ]:
# %% [markdown]
# ### CV ➜ Graph: matrice de corrélation (Spearman)

from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DfCvGraph2 = DfCvGraph[DfCvGraph["model"].str.contains("bce") & ~DfCvGraph["model"].str.contains("dice")].copy()
# --- Prépare colonnes CV & colonnes graph ---
cv_cols = [c for c in DfCvGraph2.columns if c not in ["model","metric","abs_error_mean","std_abs_error"]]
cv_cols = ["Precision", "Recall", "F1Score", "MeanIoU", "HausdorffDistance", "APLS", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]
graph_metrics = sorted(DfCvGraph2["metric"].dropna().unique().tolist())
graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule ρ et p-valeurs par (cv, graph_metric) ---
rho_rows, p_rows = [], []
for cv in cv_cols:
    rho_row, p_row = {}, {}
    for g in graph_metrics:
        d = DfCvGraph2[DfCvGraph2["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            d_cv = d[cv]
            if cv in ["HausdorffDistance", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]:
                d_cv = -d_cv # Parce que distance donc inverser les score
            rho, p = spearmanr(d_cv, -d["abs_error_mean"])
        rho_row[g] = float(rho) if np.isfinite(rho) else np.nan
        p_row[g]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)


print("Corrélation (ρ Spearman):")
display(CorrSpearman.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

print("Significativité (*<.05, **<.01, ***<.001):")

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.45*len(cv_cols)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Spearman ρ (CV vs abs_error_mean)")
plt.yticks(range(len(cv_cols)), cv_cols)
plt.xticks(range(len(graph_metrics)), graph_metrics, rotation=45, ha="right")
plt.title("BCE + Dice  only - Correlation table (Spearman) between Segmentation metrics and Graph error", pad=20)
# Annotations: ρ + étoiles
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        rho = CorrSpearman.iat[i, j]
        p = Pvalues.iat[i, j]
        if np.isnan(p):
            s = ""
        elif p < 0.001:
            s = "***"
        elif p < 0.01:
            s = "**"
        elif p < 0.05:
            s = "*"
        else:
            s = ""
        s = ""
        txt = "" if np.isnan(rho) else f"{rho:.4f}{s}"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_cv_to_graph_abs_error_spearman.png", transparent=True, dpi=600)
plt.show()

# Option: export CSV
#CorrSpearman.to_csv("corr_cv_to_graph_abs_error_spearman.csv")
#Pvalues.to_csv("corr_cv_to_graph_abs_error_pvalues.csv")


In [ ]:
DfCvGraph

In [ ]:
# DfCVGraph : val_loss = loss_DiceLoss + loss_CLDice_Dice + loss_BCE + loss_BCEDiceLoss

# one loss should have a value, others should be nan, check if this is the case
DfCvGraph3 = DfCvGraph.copy()
DfCvGraph3['num_losses'] = DfCvGraph[['loss_DiceLoss', 'loss_CLDice_Dice', 'loss_BCE', 'loss_BCEDiceLoss']].notna().sum(axis=1).copy()
print("Distribution du nombre de fonctions de perte non nulles par modèle:")
print("-----------------------------------------------")
pprint.pprint(DfCvGraph3.value_counts().to_dict())
print("-----------------------------------------------")

DfCvGraph3[DfCvGraph3['num_losses'] != 1][['model', 'loss_DiceLoss', 'loss_CLDice_Dice', 'loss_BCE', 'loss_BCEDiceLoss']]
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy.stats as stats

# Plot distribution of loss values for each loss function
loss_cols = ['loss_DiceLoss', 'loss_CLDice_Dice', 'loss_BCE', 'loss_BCEDiceLoss']
plt.figure(figsize=(12, 8))
for i, col in enumerate(loss_cols):
    plt.subplot(2, 2, i+1)
    sns.histplot(DfCvGraph3[col].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel('Loss Value')
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# DfCvGraph3['loss_val'] = sum of DfCvGraph3['loss'], if na, we add 0 
DfCvGraph['val_epoch_loss'] = DfCvGraph[loss_cols].sum(axis=1, skipna=True)
DfCvGraph

In [ ]:
import re
import numpy as np
import pandas as pd

# Copie comme tu fais déjà
_df = DfCvGraph[['model','metric','abs_error_mean','std_abs_error', 'train_epoch_loss', 'val_epoch_loss']].copy()

def _epoch_from_name(name: str) -> float:
    m = re.search(r'_(\d{2,4})(?:\D|$)', name)
    return float(m.group(1)) if m else -1

def _base_from_name(name: str) -> str:
    return name.split('_')[0] if '_' in name else name

def _loss_from_name(name: str) -> str:
    # extrait "bce", "dice", "cldice" ...
    m = re.search(r'(Unet|Segformer)_(cldice_dice|bce_dice|bce|dice|)', name)
    return m.group(1) + '_' + m.group(2) if m else "unknown"

_df['epoch'] = _df['model'].map(_epoch_from_name)
_df['base']  = _df['model'].map(_base_from_name)
_df['loss']  = _df['model'].map(_loss_from_name)

# pivot pour avoir une colonne par loss
_df_pivot = _df.pivot_table(
    index=['model','epoch','base','metric','abs_error_mean','std_abs_error'],
    columns='loss',
    values='val_epoch_loss'
).reset_index()

# renommer les colonnes en xxx_train
#_df_pivot = _df_pivot.rename(columns={c: f"{c}" for c in _df_pivot.columns if c in ["bce","dice","cldice_dice","bce_dice"]})

df_with_losses = _df_pivot

In [ ]:
df_with_losses

In [ ]:
# --- Prépare colonnes CV & colonnes graph ---
cv_cols = ["Segformer_bce", "Segformer_dice", "Segformer_bce_dice", "Unet_bce", "Unet_dice", "Unet_bce_dice", "Unet_cldice_dice"]
# ["bce", "dice", "bce_dice", "cldice_dice"]
graph_metrics = sorted(df_with_losses["metric"].dropna().unique().tolist())
graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule ρ et p-valeurs par (cv, graph_metric) ---
rho_rows, p_rows = [], []
for cv in cv_cols:
    rho_row, p_row = {}, {}
    for g in graph_metrics:
        d = df_with_losses[df_with_losses["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            d_cv = - d[cv]
            rho, p = spearmanr(d_cv, -d["abs_error_mean"])
        rho_row[g] = float(rho) if np.isfinite(rho) else np.nan
        p_row[g]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)


print("Corrélation (ρ Spearman):")
display(CorrSpearman.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

print("Significativité (*<.05, **<.01, ***<.001):")

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.45*len(cv_cols)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Spearman ρ (model val loss vs abs_error_mean)")
plt.yticks(range(len(cv_cols)), ["Val loss - " + c for c in cv_cols])
plt.xticks(range(len(graph_metrics)), graph_metrics, rotation=45, ha="right")
plt.title("Correlation table (Spearman) between model x loss validation epoch loss and Graph error", pad=20)
# Annotations: ρ + étoiles
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        rho = CorrSpearman.iat[i, j]
        p = Pvalues.iat[i, j]
        if np.isnan(p):
            s = ""
        elif p < 0.001:
            s = "***"
        elif p < 0.01:
            s = "**"
        elif p < 0.05:
            s = "*"
        else:
            s = ""
        s = ""
        txt = "" if np.isnan(rho) else f"{rho:.4f}{s}"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_cv_to_graph_abs_error_loss_model_spearman.png", transparent=True, dpi=600)
plt.show()


In [ ]:
# --- Clean, poster-ready Spearman heatmap between validation loss and graph error ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# --- Columns (as in your script) ---
cv_cols = ["Segformer_bce", "Segformer_dice", "Segformer_bce_dice",
           "Unet_bce", "Unet_dice", "Unet_cldice_dice"]

# Metrics present in df (minus a metric you excluded)
graph_metrics = sorted(df_with_losses["metric"].dropna().unique().tolist())
if "NumberOfLateralRoots" in graph_metrics:
    graph_metrics.remove("NumberOfLateralRoots")

# =============================================================================
# 1) Compute a "variation index" per metric: SRE = |pred - gt| / (|pred| + |gt|)
#    (falls back to df_with_losses['abs_error_mean'] when pred/gt not available)
# =============================================================================
use_pred_gt = all(c in df_with_losses.columns for c in ["pred", "gt"])

if use_pred_gt:
    denom = (df_with_losses["pred"].abs() + df_with_losses["gt"].abs())
    # avoid divide-by-zero
    denom = denom.replace(0, np.nan)
    df_with_losses["graph_error"] = (df_with_losses["pred"] - df_with_losses["gt"]).abs() / denom
else:
    # fallback: use provided absolute error mean if available
    if "abs_error_mean" not in df_with_losses.columns:
        raise ValueError("Neither (pred, gt) nor abs_error_mean found in df_with_losses.")
    df_with_losses["graph_error"] = df_with_losses["abs_error_mean"]

# =============================================================================
# 2) Spearman correlations: negative signs so that higher correlation = better
#    (lower val loss associated with lower graph error), consistent with your code
# =============================================================================
rho_rows, p_rows = [], []
for cv in cv_cols:
    rho_row, p_row = {}, {}
    for g in graph_metrics:
        d = df_with_losses[df_with_losses["metric"] == g][[cv, "graph_error"]].dropna()
        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(-d[cv], -d["graph_error"])
        rho_row[g] = float(rho) if np.isfinite(rho) else np.nan
        p_row[g]   = float(p) if np.isfinite(p) else np.nan
    rho_rows.append(pd.Series(rho_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrSpearman = pd.DataFrame(rho_rows)
Pvalues      = pd.DataFrame(p_rows)

# Optional: show tables in console / notebook
print("Spearman Correlation Coefficients (ρ):")
display(CorrSpearman.round(3))
print("Associated p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))
print("Significance Levels (*p* < .05, **p* < .01, ***p* < .001)")

# =============================================================================
# 3) Pretty labels for poster (English)
# =============================================================================
# Y-axis labels (validation loss variants)
cv_label_map = {
    "Segformer_bce":        "Segformer (BCE)",
    "Segformer_dice":       "Segformer (Dice)",
    "Segformer_bce_dice":   "Segformer (BCE+Dice)",
    "Unet_bce":             "UNet (BCE)",
    "Unet_dice":            "UNet (Dice)",
    "Unet_bce_dice":        "UNet (BCE+Dice)",
    "Unet_cldice_dice":     "UNet (clDice+Dice)",
}

def pretty_metric(name: str) -> str:
    # Custom replacements first
    repl = {
        "TotalRootLength": "Total Root Length",
        "PrimaryRootLength": "Primary Root Length",
        "LateralRootLength": "Lateral Root Length",
        "NumberOfOrgans": "Number of Organs",
        "RootDensity": "Root Density",
        "ConvexHullArea": "Convex Hull Area",
    }
    if name in repl:
        return repl[name]
    # Generic fallback: Title Case + spaces
    return name.replace("_", " ").replace("-", " ").title()

y_labels = [cv_label_map.get(c, f"Validation Loss – {c}") for c in cv_cols]
x_labels = [pretty_metric(g) for g in graph_metrics]

# =============================================================================
# 4) Heatmap (matplotlib) with improved title and colorbar label
# =============================================================================
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.48*len(cv_cols)+3))
im = plt.imshow(CorrSpearman.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Spearman’s ρ (Validation Loss vs. Graph Error)")

plt.yticks(range(len(cv_cols)), y_labels)
plt.xticks(range(len(graph_metrics)), x_labels, rotation=45, ha="right")

plt.title("Validation Loss vs. Graph-Level Error — Spearman Correlation", pad=20)

# --- Annotate cells: ρ + significance stars ---
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        rho = CorrSpearman.iat[i, j]
        p   = Pvalues.iat[i, j]
        if np.isnan(rho):
            txt = ""
        else:
            # significance stars
            if pd.isna(p):
                stars = ""
            elif p < 0.001:
                stars = "***"
            elif p < 0.01:
                stars = "**"
            elif p < 0.05:
                stars = "*"
            else:
                stars = ""
            stars = ''
            txt = f"{rho:.3f}{stars}"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")

plt.tight_layout()
plt.savefig("spearman_val_loss_vs_graph_error_heatmap.png", transparent=True, dpi=600)
plt.show()


In [ ]:
# --- Heatmap: each cell colored (by R² or ρ), text shows Slope | R² | ρ ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LinearRegression
import matplotlib.cm as cm

# --- Colonnes (comme dans ton code)
cv_cols = ["Segformer_bce", "Segformer_dice", "Segformer_bce_dice",
           "Unet_bce", "Unet_dice", "Unet_cldice_dice"]

graph_metrics = sorted(df_with_losses["metric"].dropna().unique().tolist())
if "NumberOfLateralRoots" in graph_metrics:
    graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule slope, R², Spearman ρ
Slope = pd.DataFrame(index=cv_cols, columns=graph_metrics, dtype=float)
R2    = pd.DataFrame(index=cv_cols, columns=graph_metrics, dtype=float)
Rho   = pd.DataFrame(index=cv_cols, columns=graph_metrics, dtype=float)

for cv in cv_cols:
    for g in graph_metrics:
        d = df_with_losses[df_with_losses["metric"] == g][[cv, "graph_error"]].dropna()
        if len(d) < 3:
            Slope.loc[cv,g] = np.nan
            R2.loc[cv,g] = np.nan
            Rho.loc[cv,g] = np.nan
            continue
        x = -d[cv].to_numpy().reshape(-1,1)
        y = -d["graph_error"].to_numpy()

        lr = LinearRegression().fit(x,y)
        slope = lr.coef_[0]
        r,p = pearsonr(x.ravel(),y)
        r2 = r**2
        rho,_ = spearmanr(x.ravel(),y)

        Slope.loc[cv,g] = slope
        R2.loc[cv,g] = r2
        Rho.loc[cv,g] = rho

# --- Labels jolis
cv_label_map = {
    "Segformer_bce":        "Segformer (BCE)",
    "Segformer_dice":       "Segformer (Dice)",
    "Segformer_bce_dice":   "Segformer (BCE+Dice)",
    "Unet_bce":             "UNet (BCE)",
    "Unet_dice":            "UNet (Dice)",
    "Unet_bce_dice":        "UNet (BCE+Dice)",
    "Unet_cldice_dice":     "UNet (clDice+Dice)",
}
def pretty_metric(name: str) -> str:
    repl = {"TotalRootLength":"Total Root Length","PrimaryRootLength":"Primary Root Length",
            "LateralRootLength":"Lateral Root Length","NumberOfOrgans":"Number of Organs",
            "RootDensity":"Root Density","ConvexHullArea":"Convex Hull Area"}
    return repl.get(name, name.replace("_"," ").title())

y_labels = [cv_label_map.get(c,c) for c in cv_cols]
x_labels = [pretty_metric(g) for g in graph_metrics]

# --- Couleur des cellules : par exemple Spearman ρ
color_metric = (Rho + R2 ) / 2  # mix des deux
values = color_metric.values.astype(float)

cmap = cm.get_cmap("RdBu_r")
vmin, vmax = -1, 1  # si Spearman ; si R² utiliser vmin=0,vmax=1
if color_metric is R2:
    vmin, vmax = 0, 1

fig, ax = plt.subplots(figsize=(1.2*len(graph_metrics)+3,0.5*len(cv_cols)+3))

im = ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

# Axes
ax.set_xticks(np.arange(len(graph_metrics)))
ax.set_yticks(np.arange(len(cv_cols)))
ax.set_xticklabels(x_labels, rotation=45, ha="right")
ax.set_yticklabels(y_labels)
ax.set_title("Validation Loss vs Graph Error — R² | Spearman ρ", pad=20)

# Texte dans chaque cellule
for i,cv in enumerate(cv_cols):
    for j,g in enumerate(graph_metrics):
        slope = Slope.loc[cv,g]
        r2    = R2.loc[cv,g]
        rho   = Rho.loc[cv,g]
        if not np.isfinite(r2) or not np.isfinite(rho):
            txt = ""
        else:
            txt = f"R²={r2:.2f}\nρ={rho:.2f}"
        color = "white" if (np.isfinite(values[i,j]) and abs(values[i,j]) >= 0.60) else "black"
        ax.text(j,i,txt,ha="center",va="center",fontsize=12,color=color)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("heatmap_slope_r2_rho.png",dpi=600,transparent=True)
plt.show()


Quantitative evaluation for the different model architectures x losses compared in this work

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy import stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Prépare colonnes CV & colonnes graph ---
cv_cols = ["Segformer_bce", "Segformer_dice", "Segformer_bce_dice", "Unet_bce", "Unet_dice", "Unet_cldice_dice"]
graph_metrics = sorted(df_with_losses["metric"].dropna().unique().tolist())
graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule R² et p-valeurs par (cv, graph_metric) ---
r2_rows, p_rows = [], []
for cv in cv_cols:
    r2_row, p_row = {}, {}
    for g in graph_metrics:
        d = df_with_losses[df_with_losses["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            r2, p = np.nan, np.nan
        else:
            d_cv = -d[cv].values.reshape(-1, 1)
            y = -d["abs_error_mean"].values
            model = LinearRegression().fit(d_cv, y)
            r2 = model.score(d_cv, y)
            # p-value via F-statistic
            n = len(y)
            if n > 2:
                y_hat = model.predict(d_cv)
                residuals = y - y_hat
                ssr = np.sum((y_hat - np.mean(y))**2)
                sse = np.sum((y - y_hat)**2)
                df_reg = 1
                df_res = n - 2
                msr = ssr / df_reg
                mse = sse / df_res if df_res > 0 else np.nan
                F = msr / mse if mse != 0 else np.nan
                p = 1 - stats.f.cdf(F, df_reg, df_res) if np.isfinite(F) else np.nan
            else:
                p = np.nan
        r2_row[g] = float(r2) if np.isfinite(r2) else np.nan
        p_row[g] = float(p) if np.isfinite(p) else np.nan
    r2_rows.append(pd.Series(r2_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrR2 = pd.DataFrame(r2_rows)
Pvalues = pd.DataFrame(p_rows)

cv_label_map = {
    "Segformer_bce":        "Segformer (BCE)",
    "Segformer_dice":       "Segformer (Dice)",
    "Segformer_bce_dice":   "Segformer (BCE+Dice)",
    "Unet_bce":             "UNet (BCE)",
    "Unet_dice":            "UNet (Dice)",
    "Unet_bce_dice":        "UNet (BCE+Dice)",
    "Unet_cldice_dice":     "UNet (clDice+Dice)",
}

def pretty_metric(name: str) -> str:
    # Custom replacements first
    repl = {
        "TotalRootLength": "Total Root Length",
        "PrimaryRootLength": "Primary Root Length",
        "LateralRootLength": "Lateral Root Length",
        "NumberOfOrgans": "Number of Organs",
        "RootDensity": "Root Density",
        "ConvexHullArea": "Convex Hull Area",
    }
    if name in repl:
        return repl[name]
    # Generic fallback: Title Case + spaces
    return name.replace("_", " ").replace("-", " ").title()

y_labels = [cv_label_map.get(c, f"Validation Loss – {c}") for c in cv_cols]
x_labels = [pretty_metric(g) for g in graph_metrics]


print("Corrélation (R² OLS):")
display(CorrR2.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

print("Significativité (*<.05, **<.01, ***<.001):")

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.45*len(cv_cols)+3))
im = plt.imshow(CorrR2.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="R² (model val loss vs abs_error_mean)")
plt.yticks(range(len(cv_cols)), y_labels)
plt.xticks(range(len(graph_metrics)), x_labels, rotation=45, ha="right")
plt.title("Correlation table (R²) etween each model validation loss and mean average error of extracted traits", pad=20)

# Annotations: R² + étoiles (couleur auto selon l'ombrage)
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        r2 = CorrR2.iat[i, j]
        p = Pvalues.iat[i, j]
        if np.isnan(p):
            s = ""
        elif p < 0.001:
            s = "***"
        elif p < 0.01:
            s = "**"
        elif p < 0.05:
            s = "*"
        else:
            s = ""
        s = ""
        txt = "" if np.isnan(r2) else f"{r2:.4f}{s}"
        # Texte blanc si |R²| élevé pour meilleur contraste
        color = "white" if (np.isfinite(r2) and abs(r2) >= 0.75) else "black"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.savefig("r2_val_loss_vs_graph_error_heatmap.png", transparent=True, dpi=600)


In [ ]:
# %% [markdown]
# ### CV ➜ Graph: matrice de corrélation (R² OLS)

from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Prépare colonnes CV & colonnes graph ---
cv_cols = [c for c in DfCvGraph.columns if c not in ["model","metric","abs_error_mean","std_abs_error"]]
cv_cols = ["Precision", "Recall", "F1Score", "MeanIoU", "HausdorffDistance", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]
graph_metrics = sorted(DfCvGraph["metric"].dropna().unique().tolist())
graph_metrics.remove("NumberOfLateralRoots")

# --- Calcule R² et p-valeurs par (cv, graph_metric) ---
r2_rows, p_rows = [], []
for cv in cv_cols:
    r2_row, p_row = {}, {}
    for g in graph_metrics:
        d = DfCvGraph[DfCvGraph["metric"] == g][[cv, "abs_error_mean"]].dropna()
        if len(d) < 3:
            r2, p = np.nan, np.nan
        else:
            d_cv = d[cv].values.reshape(-1, 1)
            y = -d["abs_error_mean"].values
            if cv in ["HausdorffDistance", "AverageCenterlineDistance", "Betti0VariationIndexGPU", "Betti1VariationIndexGPU"]:
                d_cv = -d_cv # Parce que distance donc inverser les score
            model = LinearRegression().fit(d_cv, y)
            r2 = model.score(d_cv, y)
            # p-value via F-statistic
            from scipy import stats
            n = len(y)
            if n > 2:
                y_hat = model.predict(d_cv)
                residuals = y - y_hat
                ssr = np.sum((y_hat - np.mean(y))**2)
                sse = np.sum((y - y_hat)**2)
                df_reg = 1
                df_res = n - 2
                msr = ssr / df_reg
                mse = sse / df_res if df_res > 0 else np.nan
                F = msr / mse if mse != 0 else np.nan
                p = 1 - stats.f.cdf(F, df_reg, df_res) if np.isfinite(F) else np.nan
            else:
                p = np.nan
        r2_row[g] = float(r2) if np.isfinite(r2) else np.nan
        p_row[g] = float(p) if np.isfinite(p) else np.nan
    r2_rows.append(pd.Series(r2_row, name=cv))
    p_rows.append(pd.Series(p_row, name=cv))

CorrR2 = pd.DataFrame(r2_rows)
Pvalues = pd.DataFrame(p_rows)

print("Corrélation (R² OLS):")
display(CorrR2.round(3))

print("p-values:")
display(Pvalues.applymap(lambda x: round(x, 3) if pd.notna(x) else x))

print("Significativité (*<.05, **<.01, ***<.001):")

# --- Heatmap simple (matplotlib) centrée sur 0 ---
plt.figure(figsize=(1.2*len(graph_metrics)+3, 0.45*len(cv_cols)+3))
im = plt.imshow(CorrR2.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04, label="R² (CV vs abs_error_mean)")
plt.yticks(range(len(cv_cols)), cv_cols)
plt.xticks(range(len(graph_metrics)), graph_metrics, rotation=45, ha="right")
plt.title("Correlation table (R² OLS) between Segmentation metrics and Graph error", pad=20)
# Annotations: R² + étoiles
for i in range(len(cv_cols)):
    for j in range(len(graph_metrics)):
        r2 = CorrR2.iat[i, j]
        p = Pvalues.iat[i, j]
        if np.isnan(p):
            s = ""
        elif p < 0.001:
            s = "***"
        elif p < 0.01:
            s = "**"
        elif p < 0.05:
            s = "*"
        else:
            s = ""
        s = ""
        txt = "" if np.isnan(r2) else f"{r2:.4f}{s}"
        plt.text(j, i, txt, ha='center', va='center', fontsize=8, color="black")
plt.tight_layout()
plt.savefig("corr_cv_to_graph_abs_error_r2.png", transparent=True, dpi=600)
plt.show()


In [ ]:
DfCvGraph

In [ ]:
# --- Imports ---
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def base_label(model_name: str) -> str:
    """
    Extrait un label 'modèle_loss' sans suffixe d'époque.
    Ex.: 'Unet_bce_200' -> 'Unet_bce' ; 'unet_cldice_dice_ep120' -> 'unet_cldice_dice'
    """
    s = str(model_name)
    s = re.sub(r'(_ep?\d{1,5})\b.*$', '', s, flags=re.IGNORECASE)  # _ep123 / _123
    s = re.sub(r'(_\d{1,5})\b.*$', '', s, flags=re.IGNORECASE)
    return s

def make_color_map(labels):
    """
    Une couleur distincte par label (modèle×loss).
    Utilise tab20/tab20b/tab20c (60 couleurs). Si >60, fallback HSV.
    """
    labels = list(labels)
    n = len(labels)

    cols = []
    for cmap_name in ["tab20", "tab20b", "tab20c"]:
        cmap = plt.get_cmap(cmap_name)
        cols.extend(list(getattr(cmap, "colors", [])))

    if n <= len(cols):
        palette = cols[:n]
    else:
        hsv = plt.get_cmap("hsv")
        palette = [hsv(i / n) for i in range(n)]

    return {lab: palette[i] for i, lab in enumerate(labels)}

# ------------------------------------------------------------
# Figure : Hausdorff vs Absolute Error (NumberOfOrgans) — sans régressions
# ------------------------------------------------------------
def plot_hausdorff_vs_numorg(
    df,
    x_metric="HausdorffDistance",
    y_metric="abs_error_mean",
    trait_key="NumberOfOrgans",
    labels_inline=True,           # annotations près des points
    annotate_epoch_only=True,     # si True: n’annoter que l’epoch (si dispo)
    show_errorbar=False,          # barres d’erreur si 'std_abs_error' dispo
    legend_ncol=1,                # nb de colonnes pour la légende
    save_basename="figure_hausdorff_vs_numorg",
    dpi=600
):
    """
    X = Hausdorff distance ; Y = absolute error (Number of Organs)
    - Couleur = modèle×loss (ex: Unet_bce), identique pour toutes ses époques
    - Légende = un item par modèle×loss, sans préciser l’époque (pas de régressions)
    - Sauvegarde PNG/PDF/SVG
    """
    required_cols = {"model", "metric", x_metric, y_metric}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes dans df: {missing}")

    d = df.dropna(subset=[x_metric, "metric", y_metric]).copy()
    d = d[d["metric"] == trait_key]
    if d.empty:
        raise ValueError(f"Aucune donnée pour metric == '{trait_key}'.")

    # Label de groupe (modèle×loss sans époque)
    d["base_label"] = d["model"].map(base_label)
    base_labels = sorted(d["base_label"].unique().tolist())
    color_map = make_color_map(base_labels)
    marker_map = {bl: "o" for bl in base_labels}  # même marqueur (modifiable)

    # --- Figure ---
    plt.figure(figsize=(11, 7))
    ax = plt.gca()

    # Scatter par groupe (sans label pour éviter d'encombrer la légende)
    for bl in base_labels:
        dg = d[d["base_label"] == bl]
        x = dg[x_metric].values
        y = dg[y_metric].values
        c = color_map[bl]
        mk = marker_map[bl]

        ax.scatter(
            x, y, s=60, alpha=0.95, label=None, color=c, marker=mk,
            edgecolor="white", linewidth=0.6
        )

        if show_errorbar and ("std_abs_error" in dg.columns):
            ax.errorbar(
                x, y, yerr=dg["std_abs_error"].values, fmt="none",
                ecolor=c, alpha=0.45, capsize=2
            )

        # Étiquettes inline (points individuels)
        if labels_inline:
            x_min, x_max = ax.get_xlim(); y_min, y_max = ax.get_ylim()
            dx = 0.012*(x_max-x_min); dy = 0.012*(y_max-y_min)
            for _, row in dg.iterrows():
                xi, yi = row[x_metric], row[y_metric]
                model_full = str(row["model"])
                if annotate_epoch_only:
                    m = re.search(r'(?:_ep)?(\d{1,5})\b', model_full, flags=re.IGNORECASE)
                    label_txt = m.group(1) if m else ""  # si pas d’epoch, pas d'annotation
                else:
                    label_txt = model_full
                if label_txt:
                    ax.annotate(
                        label_txt, (xi, yi), xytext=(dx, dy), textcoords="offset points",
                        fontsize=8, color="black",
                        path_effects=[pe.withStroke(linewidth=2.2, foreground="white", alpha=0.9)]
                    )

    # Habillage
    ax.set_title("Absolute error (Number of Organs) vs Hausdorff distance")
    ax.set_xlabel("Hausdorff distance (lower is better)")
    ax.set_ylabel("Absolute error — Number of Organs (lower is better)")
    ax.grid(True, linestyle=":", alpha=0.5)

    # Légende propre : un handle par modèle×loss (pas de points individuels, pas de régressions)
    legend_handles = [
        Line2D([0], [0], marker=marker_map[bl], linestyle="None",
               markerfacecolor=color_map[bl], markeredgecolor="white",
               markeredgewidth=0.6, markersize=8, label=bl)
        for bl in base_labels
    ]
    ax.legend(handles=legend_handles, title="Model × loss", loc="upper left",
              fontsize=8, ncol=legend_ncol)

    # Marges
    xpad = 0.03*(d[x_metric].max() - d[x_metric].min())
    ypad = 0.05*(d[y_metric].max() - d[y_metric].min())
    ax.set_xlim(d[x_metric].min() - xpad, d[x_metric].max() + xpad)
    ax.set_ylim(d[y_metric].min() - ypad, d[y_metric].max() + ypad)

    plt.tight_layout()

    # Sauvegardes
    for ext in ("png", "pdf", "svg"):
        plt.savefig(f"{save_basename}.{ext}", dpi=(dpi if ext=="png" else None),
                    bbox_inches="tight", transparent=True)
    plt.show()

    print(f"Saved: {save_basename}.png / .pdf / .svg")

# ------------------------------------------------------------
# Exemple d'appel
# ------------------------------------------------------------
plot_hausdorff_vs_numorg(
    df_scatter,
    x_metric="HausdorffDistance",
    y_metric="abs_error_mean",
    trait_key="NumberOfOrgans",
    labels_inline=False,
    annotate_epoch_only=False,
    show_errorbar=False,
    legend_ncol=1,
    save_basename="figure_hausdorff_vs_numorg",
    dpi=600
)


# Pareto